# Corso di Modelli Statistici
## Applicato ai dati NBA Shot Logs (2004–2025)

> **Dataset**: ~4.45 milioni di tiri NBA registrati stagione per stagione.  
> **Variabili chiave**: `SHOT_MADE`, `SHOT_DISTANCE`, `SHOT_TYPE`, `QUARTER`, `LOC_X/Y`, `ACTION_TYPE`, `SEASON_1`, `POSITION_GROUP`

---

In [ ]:
# ── Configurazione globale ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import binom, norm, poisson

# Stile grafico uniforme per tutto il corso
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

print('Librerie caricate correttamente.')

In [ ]:
# ── Caricamento del dataset ─────────────────────────────────────────────────
# Utilizziamo il dataset aggregato di tutte le stagioni
df = pd.read_csv('data/shots_all_seasons.csv')

# Normalizzazione SHOT_MADE → booleano
df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)  # 1 = canestro, 0 = errore

print(f'Dataset caricato: {df.shape[0]:,} righe × {df.shape[1]} colonne')
print(f'Stagioni: {df["SEASON_1"].min()} – {df["SEASON_1"].max()}')
print(f'\nTiri totali per esito:')
print(df['SHOT_MADE'].value_counts())
print(f'\nField Goal % globale: {df["SHOT_MADE"].mean():.4f}')

---
# DISPENSA 1
# Distribuzioni di Probabilità: Binomiale, CDF, Approssimazioni e Z-Score

---

## 1.1 — L'esperimento di Bernoulli

Un **esperimento di Bernoulli** è un esperimento aleatorio con esattamente due esiti possibili:
- **Successo** (con probabilità $p$)
- **Insuccesso** (con probabilità $1 - p = q$)

La **variabile aleatoria di Bernoulli** $X \sim \text{Ber}(p)$ vale:
$$
P(X = x) = p^x (1-p)^{1-x}, \quad x \in \{0, 1\}
$$

**Nel nostro contesto NBA:** ogni tiro tentato è un esperimento di Bernoulli.  
- **Successo** ($X=1$): il tiro viene realizzato (`SHOT_MADE = TRUE`)  
- **Insuccesso** ($X=0$): il tiro non viene realizzato (`SHOT_MADE = FALSE`)  
- Il parametro $p$ è la **Field Goal Percentage (FG%)**

In [ ]:
# ── 1.1 Esperimento di Bernoulli ───────────────────────────────────────────
# Calcolo del parametro p stimato dai dati
p_globale = df['SHOT_MADE'].mean()
p_2pt = df[df['SHOT_TYPE'] == '2PT Field Goal']['SHOT_MADE'].mean()
p_3pt = df[df['SHOT_TYPE'] == '3PT Field Goal']['SHOT_MADE'].mean()

print('=== Stima del parametro p (probabilità di successo) ===')
print(f'FG%  globale : p = {p_globale:.4f}  ({p_globale*100:.2f}%)')
print(f'FG%  2 punti : p = {p_2pt:.4f}  ({p_2pt*100:.2f}%)')
print(f'FG%  3 punti : p = {p_3pt:.4f}  ({p_3pt*100:.2f}%)')

# Visualizzazione della distribuzione di Bernoulli
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (label, p) in zip(axes, [('Globale', p_globale),
                                   ('2PT', p_2pt),
                                   ('3PT', p_3pt)]):
    ax.bar([0, 1], [1 - p, p], color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Errore (0)', 'Canestro (1)'])
    ax.set_ylabel('Probabilità')
    ax.set_title(f'Bernoulli({label})\np = {p:.3f}')
    ax.set_ylim(0, 1)
    for i, v in enumerate([1 - p, p]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Distribuzione di Bernoulli — Tiri NBA (2004–2025)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Interpretazione:** La FG% globale dell'NBA è circa il 45–46%, con i tiri da 2 punti attorno al 51% e quelli da 3 attorno al 36%. Ogni tiro è un lancio di moneta con dado truccato: il canestro è *meno* probabile dell'errore nella media del campionato.

---

## 1.2 — La Distribuzione Binomiale

Se ripetiamo un esperimento di Bernoulli $n$ volte in modo **indipendente**, il numero totale di successi $X$ segue una **distribuzione Binomiale**:

$$
X \sim \text{Bin}(n, p)
$$

La **funzione di massa di probabilità (PMF)** è:

$$
P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n
$$

dove $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ è il **coefficiente binomiale**.

**Valore atteso e varianza:**
$$
E[X] = np, \qquad \text{Var}(X) = np(1-p)
$$

**Contesto NBA:** Se un giocatore tenta $n = 20$ tiri in una partita con una FG% di $p$, quanti canestri ci aspettiamo che faccia?

In [ ]:
# ── 1.2 Distribuzione Binomiale ────────────────────────────────────────────
n = 20      # tiri tentati in una partita
p  = p_2pt  # FG% da 2 punti stimata dai dati

k_vals = np.arange(0, n + 1)
pmf_vals = binom.pmf(k_vals, n, p)

media    = n * p
varianza = n * p * (1 - p)
std      = np.sqrt(varianza)

print(f'Parametri: n = {n}, p = {p:.4f}')
print(f'Valore atteso E[X] = n·p = {media:.2f} canestri')
print(f'Varianza Var(X) = n·p·(1-p) = {varianza:.2f}')
print(f'Dev. Standard σ = {std:.2f}')
print(f'\nP(X = 10) = {binom.pmf(10, n, p):.4f}  → prob. di fare esattamente 10 canestri su 20 tiri')
print(f'P(X ≥ 12) = {1 - binom.cdf(11, n, p):.4f}  → prob. di fare almeno 12 canestri su 20 tiri')

# Grafico PMF
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#3498db' if abs(k - media) <= std else '#aed6f1' for k in k_vals]
ax.bar(k_vals, pmf_vals, color=colors, edgecolor='black', linewidth=0.6)
ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'E[X] = {media:.1f}')
ax.axvline(media - std, color='orange', linestyle=':', linewidth=1.5, label=f'±1σ = [{media-std:.1f}, {media+std:.1f}]')
ax.axvline(media + std, color='orange', linestyle=':', linewidth=1.5)
ax.set_xlabel('k (numero di canestri su 20 tiri)')
ax.set_ylabel('P(X = k)')
ax.set_title(f'PMF — Bin(n={n}, p={p:.3f}) | FG% da 2 punti NBA')
ax.legend()
ax.set_xticks(k_vals)

# Annotazione
k_max = np.argmax(pmf_vals)
ax.annotate(f'Moda: k={k_max}\nP={pmf_vals[k_max]:.3f}',
            xy=(k_max, pmf_vals[k_max]),
            xytext=(k_max + 2, pmf_vals[k_max] + 0.01),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)
plt.tight_layout()
plt.show()

**Interpretazione:** Con una FG% da 2 punti di circa 51%, un giocatore che tenta 20 tiri si aspetta di realizzarne circa 10. Il grafico mostra la distribuzione a campana centrata sul valore atteso, con le barre blu scuro che indicano i valori entro una deviazione standard dalla media.

---

## 1.3 — La Funzione di Ripartizione (CDF)

La **Funzione di Ripartizione** (o Cumulative Distribution Function, CDF) risponde alla domanda:

> *"Qual è la probabilità che X sia al massimo k?"*

$$
F(k) = P(X \leq k) = \sum_{i=0}^{k} \binom{n}{i} p^i (1-p)^{n-i}
$$

**Proprietà fondamentali:**
- $F(-\infty) = 0$ e $F(+\infty) = 1$
- $F$ è non decrescente
- $P(a < X \leq b) = F(b) - F(a)$

Grazie alla CDF possiamo rispondere a domande del tipo:  
*"Quale la probabilità che un giocatore faccia MENO di 8 canestri su 20 tiri?"*

In [ ]:
# ── 1.3 Funzione di Ripartizione (CDF) ────────────────────────────────────
cdf_vals = binom.cdf(k_vals, n, p)

# Domande pratiche
q1 = binom.cdf(7, n, p)          # P(X <= 7)
q2 = 1 - binom.cdf(13, n, p)     # P(X > 13)
q3 = binom.cdf(12, n, p) - binom.cdf(7, n, p)  # P(8 <= X <= 12)

print('=== Domande pratiche con la CDF ===')
print(f'P(X ≤  7) = {q1:.4f}  → meno di 8 canestri su 20 tiri')
print(f'P(X > 13) = {q2:.4f}  → più di 13 canestri (prestazione eccezionale)')
print(f'P(8 ≤ X ≤ 12) = {q3:.4f}  → tra 8 e 12 canestri (intervallo "normale")')

# Grafico PMF + CDF affiancati
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PMF con area shaded
colors_cdf = ['#e74c3c' if k <= 7 else '#2ecc71' if k > 13 else '#3498db' for k in k_vals]
ax1.bar(k_vals, pmf_vals, color=colors_cdf, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('k')
ax1.set_ylabel('P(X = k)')
ax1.set_title('PMF con regioni evidenziate')
ax1.set_xticks(k_vals)
patches = [
    mpatches.Patch(color='#e74c3c', label=f'P(X≤7) = {q1:.3f}'),
    mpatches.Patch(color='#3498db', label=f'P(8≤X≤12) = {q3:.3f}'),
    mpatches.Patch(color='#2ecc71', label=f'P(X>13) = {q2:.3f}'),
]
ax1.legend(handles=patches, fontsize=9)

# CDF
ax2.step(k_vals, cdf_vals, where='post', color='#2c3e50', linewidth=2.5, label='CDF F(k)')
ax2.scatter(k_vals, cdf_vals, color='#e74c3c', zorder=5, s=50)
ax2.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50° percentile')
ax2.axhline(0.25, color='orange', linestyle=':', linewidth=1, label='25° percentile')
ax2.axhline(0.75, color='green', linestyle=':', linewidth=1, label='75° percentile')
ax2.set_xlabel('k')
ax2.set_ylabel('F(k) = P(X ≤ k)')
ax2.set_title('CDF — Bin(n=20, p={:.3f})'.format(p))
ax2.set_ylim(-0.05, 1.05)
ax2.legend(fontsize=9)
ax2.set_xticks(k_vals)

plt.suptitle('PMF e CDF della Distribuzione Binomiale (FG% da 2pt NBA)', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretazione:** La CDF cresce in modo a gradini (è una variabile discreta). Si noti che il 50° percentile cade circa a 10 canestri, coerentemente con il valore atteso. La zona rossa indica prestazioni al di sotto della media attesa, la verde quelle eccezionali.

---

## 1.4 — Approssimazione Normale della Binomiale

Per $n$ sufficientemente grande, la distribuzione Binomiale può essere approssimata con una **distribuzione Normale**:

$$
\text{Bin}(n, p) \approx \mathcal{N}(\mu, \sigma^2) \quad \text{se} \quad np \geq 5 \; \text{e} \; n(1-p) \geq 5
$$

dove $\mu = np$ e $\sigma^2 = np(1-p)$.

**Correzione di continuità:** Poiché passiamo da una variabile discreta a una continua, si applica:
$$
P(X = k) \approx P\left(k - 0.5 \leq Y \leq k + 0.5\right), \quad Y \sim \mathcal{N}(\mu, \sigma^2)
$$

**Contesto NBA:** Consideriamo una squadra che tenta $n = 90$ tiri in una partita — quanti ne farà?

In [ ]:
# ── 1.4 Approssimazione Normale della Binomiale ────────────────────────────
# FG% media per una squadra (tutti i tiri)
n_sq = 90   # tiri tentati in una partita (realistico per una squadra NBA)
p_sq = p_globale

mu    = n_sq * p_sq
sigma = np.sqrt(n_sq * p_sq * (1 - p_sq))

print(f'Parametri: n = {n_sq}, p = {p_sq:.4f}')
print(f'μ = n·p = {mu:.2f},  σ = √(n·p·(1-p)) = {sigma:.2f}')
print(f'Condizioni per approssimazione: np = {mu:.1f} ≥ 5 ✓   n(1-p) = {n_sq*(1-p_sq):.1f} ≥ 5 ✓')

k_vals2 = np.arange(25, 65)
pmf2    = binom.pmf(k_vals2, n_sq, p_sq)

x_cont  = np.linspace(25, 65, 400)
pdf_norm = norm.pdf(x_cont, mu, sigma)

# Confronto grafico
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(k_vals2, pmf2, color='#aed6f1', edgecolor='#2980b9', linewidth=0.6,
       label=f'Bin({n_sq}, {p_sq:.3f}) — esatta', alpha=0.85)
ax.plot(x_cont, pdf_norm, color='#e74c3c', linewidth=2.5,
        label=f'N(μ={mu:.1f}, σ={sigma:.2f}) — appross.')
ax.axvline(mu, color='darkred', linestyle='--', linewidth=1.5, label=f'μ = {mu:.1f}')
ax.fill_between(x_cont,
                pdf_norm,
                where=(x_cont >= mu - sigma) & (x_cont <= mu + sigma),
                alpha=0.2, color='#e74c3c', label=f'μ ± σ  ≈ 68%')
ax.set_xlabel('Canestri realizzati su 90 tiri')
ax.set_ylabel('Probabilità / Densità')
ax.set_title(f'Approssimazione Normale della Binomiale — Squadra NBA (90 tiri/partita)')
ax.legend()
plt.tight_layout()
plt.show()

# Verifica numerica dell'approssimazione
k_test = 42  # canestri
esatta  = binom.pmf(k_test, n_sq, p_sq)
approx  = norm.pdf(k_test, mu, sigma)
approx_cc = norm.cdf(k_test + 0.5, mu, sigma) - norm.cdf(k_test - 0.5, mu, sigma)
print(f'\nP(X={k_test}) esatta        = {esatta:.5f}')
print(f'P(X={k_test}) approx. Normale = {approx:.5f}')
print(f'P(X={k_test}) con corr. cont.  = {approx_cc:.5f}   ← più precisa')

**Interpretazione:** La curva normale (rossa) si sovrappone quasi perfettamente alle barre della binomiale, dimostrando l'efficacia dell'approssimazione per $n$ grande. La correzione di continuità (+0.5) rende il risultato ancora più preciso.

---

## 1.5 — Approssimazione di Poisson

La **distribuzione di Poisson** approssima la Binomiale quando $n$ è molto grande e $p$ è molto piccolo (eventi rari), con $\lambda = np$ finito:

$$
\text{Bin}(n, p) \approx \text{Pois}(\lambda) \quad \text{se} \quad n \to \infty,\; p \to 0,\; np = \lambda \text{ costante}
$$

La PMF di Poisson è:
$$
P(X = k) = \frac{e^{-\lambda} \lambda^k}{k!}, \quad k = 0, 1, 2, \ldots
$$

**Valore atteso e varianza:**
$$
E[X] = \text{Var}(X) = \lambda
$$

**Contesto NBA:** Consideriamo gli **Alley Oop Dunk** — un tipo di tiro molto spettacolare ma relativamente raro (probabilità bassa). In una partita con ~90 tiri totali, quanti Alley Oop ci aspettiamo?

In [ ]:
# ── 1.5 Approssimazione di Poisson ────────────────────────────────────────
# Identifichiamo i tipi di tiro rari nel dataset
shot_counts = df['ACTION_TYPE'].value_counts(normalize=True)
print('Top 5 tipi di tiro più comuni:')
print(shot_counts.head(5).to_string())
print('\nTipi di tiro rari (< 0.5% delle occorrenze):')
rari = shot_counts[shot_counts < 0.005]
print(rari.head(8).to_string())

# Alley Oop Dunk
p_alleyoop = shot_counts.get('Alley Oop Dunk Shot', 0)
n_tiri = 90  # tiri per partita (una squadra)
lam = n_tiri * p_alleyoop  # lambda = n*p

print(f'\n=== Alley Oop Dunk Shot ===')
print(f'Proporzione nel dataset: p = {p_alleyoop:.5f}')
print(f'λ = n·p = {n_tiri} × {p_alleyoop:.5f} = {lam:.4f}')
print(f'Condizione: n grande ({n_tiri}), p piccolo ({p_alleyoop:.5f}) ✓')

In [ ]:
# ── Confronto Binomiale vs Poisson per evento raro ─────────────────────────
# Usiamo un tiro raro con p misurabile: Alley Oop Layup Shot
p_rare = shot_counts.get('Alley Oop Layup Shot', None)
if p_rare is None:
    # fallback: usiamo un valore tipico per eventi rari
    tipo_raro = shot_counts[shot_counts < 0.01].index[0]
    p_rare = shot_counts[tipo_raro]
else:
    tipo_raro = 'Alley Oop Layup Shot'

# Scenario: 200 tiri totali in una partita (entrambe le squadre)
n_partita = 200
p_ev = p_rare
lam_ev = n_partita * p_ev

k_range = np.arange(0, 15)
pmf_binom_rare = binom.pmf(k_range, n_partita, p_ev)
pmf_poisson    = poisson.pmf(k_range, lam_ev)

print(f'Evento raro: "{tipo_raro}"')
print(f'p = {p_ev:.5f},  n = {n_partita},  λ = {lam_ev:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

width = 0.35
ax1.bar(k_range - width/2, pmf_binom_rare, width, color='#3498db',
        edgecolor='black', linewidth=0.5, label=f'Bin({n_partita}, {p_ev:.4f})', alpha=0.85)
ax1.bar(k_range + width/2, pmf_poisson, width, color='#e67e22',
        edgecolor='black', linewidth=0.5, label=f'Pois(λ={lam_ev:.3f})', alpha=0.85)
ax1.set_xlabel('k (numero di eventi)')
ax1.set_ylabel('P(X = k)')
ax1.set_title('PMF: Binomiale vs Poisson\n(evento raro)')
ax1.legend()
ax1.set_xticks(k_range)

# Errore relativo
err_rel = np.abs(pmf_binom_rare - pmf_poisson) / (pmf_binom_rare + 1e-12)
ax2.bar(k_range, err_rel * 100, color='#9b59b6', edgecolor='black', linewidth=0.5)
ax2.set_xlabel('k')
ax2.set_ylabel('Errore relativo (%)')
ax2.set_title('Errore relativo dell\'approssimazione\n|Bin − Pois| / Bin')
ax2.set_xticks(k_range)

plt.suptitle(f'Approssimazione di Poisson — "{tipo_raro}"', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nErrore relativo medio: {np.mean(err_rel[pmf_binom_rare > 1e-6])*100:.2f}%')
print('→ Le due distribuzioni sono quasi identiche: l\'approssimazione è eccellente.')

**Interpretazione:** L'approssimazione di Poisson funziona eccellentemente per eventi rari (errore relativo medio tipicamente sotto il 2%). Questo è utile in pratica perché la PMF di Poisson è molto più semplice da calcolare rispetto alla Binomiale per $n$ grandi.

---

## 1.6 — Z-Score e Standardizzazione

Lo **Z-score** (o punteggio standard) misura quante deviazioni standard un valore $x$ dista dalla media $\mu$:

$$
Z = \frac{X - \mu}{\sigma}
$$

Per una **variabile Normale** $X \sim \mathcal{N}(\mu, \sigma^2)$, la standardizzazione produce:
$$
Z = \frac{X - \mu}{\sigma} \sim \mathcal{N}(0, 1)
$$

La **Regola empirica** (per distribuzioni normali) afferma:
$$
P(|Z| \leq 1) \approx 68\%, \quad P(|Z| \leq 2) \approx 95\%, \quad P(|Z| \leq 3) \approx 99.7\%
$$

**Contesto NBA:** Utilizziamo `SHOT_DISTANCE` per identificare i tiri "anomali" rispetto alla media.

In [ ]:
# ── 1.6 Z-Score sulla variabile SHOT_DISTANCE ─────────────────────────────
dist = df['SHOT_DISTANCE'].dropna().astype(float)

mu_dist  = dist.mean()
std_dist = dist.std(ddof=1)  # deviazione standard campionaria

# Calcolo Z-score
z_scores = (dist - mu_dist) / std_dist

print(f'SHOT_DISTANCE — Statistiche descrittive:')
print(f'  Media (μ):            {mu_dist:.2f} piedi')
print(f'  Dev. Standard (σ):    {std_dist:.2f} piedi')
print(f'  Min:                  {dist.min():.1f} piedi')
print(f'  Max:                  {dist.max():.1f} piedi')

print(f'\nRegola empirica (verificata sui dati):')
print(f'  |Z| ≤ 1  → {(np.abs(z_scores) <= 1).mean()*100:.1f}%  (atteso: ~68%)')
print(f'  |Z| ≤ 2  → {(np.abs(z_scores) <= 2).mean()*100:.1f}%  (atteso: ~95%)')
print(f'  |Z| ≤ 3  → {(np.abs(z_scores) <= 3).mean()*100:.1f}%  (atteso: ~99.7%)')

# Tiri anomali (|Z| > 3 = oltre ~34 piedi dalla media)
outlier_threshold = 3
outliers = df[np.abs(z_scores) > outlier_threshold][['PLAYER_NAME', 'SHOT_DISTANCE',
                                                      'SHOT_TYPE', 'ACTION_TYPE']]
print(f'\nTiri con |Z| > 3 (outlier estremi): {len(outliers):,}')
print(outliers.sort_values('SHOT_DISTANCE', ascending=False).head(5).to_string(index=False))

In [ ]:
# ── Visualizzazione Z-Score ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Distribuzione originale SHOT_DISTANCE
sample = dist.sample(50000, random_state=42)  # sample per velocità
axes[0].hist(sample, bins=60, color='#3498db', edgecolor='white', linewidth=0.3, density=True)
axes[0].axvline(mu_dist, color='red', linestyle='--', linewidth=2, label=f'μ = {mu_dist:.1f}')
axes[0].axvline(mu_dist + std_dist, color='orange', linestyle=':', linewidth=1.5, label=f'±σ')
axes[0].axvline(mu_dist - std_dist, color='orange', linestyle=':', linewidth=1.5)
axes[0].set_xlabel('Distanza (piedi)')
axes[0].set_ylabel('Densità')
axes[0].set_title('Distribuzione SHOT_DISTANCE')
axes[0].legend(fontsize=9)

# 2. Distribuzione Z-score
z_sample = z_scores[dist.index.isin(sample.index)]
x_norm = np.linspace(-5, 5, 300)
axes[1].hist(z_sample, bins=60, color='#2ecc71', edgecolor='white', linewidth=0.3, density=True)
axes[1].plot(x_norm, norm.pdf(x_norm), 'r-', linewidth=2.5, label='N(0,1) teorica')
# Colorare le regioni 1-2-3 sigma
for i, (lo, hi, col, lab) in enumerate([
    (-1, 1, '#e74c3c', '68%'), (-2, 2, '#e67e22', '95%'), (-3, 3, '#f1c40f', '99.7%')]):
    axes[1].fill_between(x_norm, norm.pdf(x_norm),
                          where=(x_norm >= lo) & (x_norm <= hi),
                          alpha=0.15, color=col, label=f'|Z|≤{abs(lo)}: {lab}')
axes[1].set_xlabel('Z-score')
axes[1].set_ylabel('Densità')
axes[1].set_title('Z-score standardizzato')
axes[1].legend(fontsize=8)
axes[1].set_xlim(-5, 5)

# 3. Q-Q Plot per verificare la normalità
z_samp_arr = z_sample.values
n_qq = min(5000, len(z_samp_arr))
qq_sample = np.random.choice(z_samp_arr, n_qq, replace=False)
(osservati, teorici), (slope, intercept, r) = stats.probplot(qq_sample, dist='norm')
axes[2].scatter(teorici, osservati, s=3, alpha=0.4, color='#3498db', label='Dati')
x_line = np.array([teorici.min(), teorici.max()])
axes[2].plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label='Linea teorica')
axes[2].set_xlabel('Quantili teorici N(0,1)')
axes[2].set_ylabel('Quantili osservati')
axes[2].set_title(f'Q-Q Plot  (r = {r:.4f})')
axes[2].legend(fontsize=9)

plt.suptitle('Standardizzazione Z-score — SHOT_DISTANCE NBA', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretazione del Q-Q Plot:** Se i punti seguissero perfettamente la linea rossa, la distribuzione sarebbe esattamente normale. La deviazione nelle code (code più pesanti) indica che `SHOT_DISTANCE` non è perfettamente normale — ha una distribuzione bimodale (picco sui tiri vicini al canestro e picco sui tiri da 3 punti). Questo è un esempio reale di come il Q-Q plot diagnostica la normalità.

---

## 1.7 — Riepilogo: Z-Score per confronto tra categorie

Lo Z-score permette di **confrontare grandezze su scale diverse**. Possiamo usarlo per confrontare le prestazioni dei giocatori in un contesto normalizzato.

In [ ]:
# ── 1.7 Z-Score per confronto tra giocatori ────────────────────────────────
# FG% per giocatore (stagione 2024), almeno 200 tiri tentati
df_2024 = df[df['SEASON_1'] == 2024]

player_stats = (df_2024.groupby('PLAYER_NAME')
                .agg(tentativi=('SHOT_MADE', 'count'),
                     canestri=('SHOT_MADE_INT', 'sum'))
                .query('tentativi >= 200')
                .assign(fg_pct=lambda x: x['canestri'] / x['tentativi']))

# Z-score della FG%
mu_fg  = player_stats['fg_pct'].mean()
std_fg = player_stats['fg_pct'].std(ddof=1)
player_stats['z_fg'] = (player_stats['fg_pct'] - mu_fg) / std_fg

print(f'Giocatori analizzati (≥200 tiri, stagione 2024): {len(player_stats)}')
print(f'FG% media lega:    {mu_fg:.3f}  ({mu_fg*100:.1f}%)')
print(f'Deviazione std:    {std_fg:.4f}')

# Top e Bottom per Z-score
top5 = player_stats.nlargest(5, 'z_fg')[['fg_pct', 'tentativi', 'z_fg']]
bot5 = player_stats.nsmallest(5, 'z_fg')[['fg_pct', 'tentativi', 'z_fg']]
print(f'\nTop 5 FG% (Z-score più alto):')
print(top5.to_string())
print(f'\nBottom 5 FG% (Z-score più basso):')
print(bot5.to_string())

# Grafico
fig, ax = plt.subplots(figsize=(12, 5))
colors_z = ['#e74c3c' if z > 2 else '#e67e22' if z > 1 else
            '#2ecc71' if z < -1 else '#95a5a6'
            for z in player_stats['z_fg']]
sorted_stats = player_stats.sort_values('z_fg')
colors_sorted = ['#e74c3c' if z > 2 else '#e67e22' if z > 1 else
                 '#2ecc71' if z < -1 else '#95a5a6'
                 for z in sorted_stats['z_fg']]
ax.bar(range(len(sorted_stats)), sorted_stats['z_fg'],
       color=colors_sorted, edgecolor='none', width=1)
ax.axhline(0, color='black', linewidth=1.5)
ax.axhline(2, color='red', linestyle='--', linewidth=1, alpha=0.7, label='|Z| = 2')
ax.axhline(-2, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(1, color='orange', linestyle=':', linewidth=1, alpha=0.7, label='|Z| = 1')
ax.axhline(-1, color='orange', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xlabel('Giocatori (ordinati per Z-score)')
ax.set_ylabel('Z-score FG%')
ax.set_title('Z-score della FG% per giocatore — Stagione NBA 2023-24 (≥200 tiri)')
ax.set_xticks([])

patches = [
    mpatches.Patch(color='#e74c3c', label='Z > 2  (top performer estremi)'),
    mpatches.Patch(color='#e67e22', label='Z > 1  (sopra media)'),
    mpatches.Patch(color='#95a5a6', label='|Z| ≤ 1 (nella norma)'),
    mpatches.Patch(color='#2ecc71', label='Z < -1 (sotto media)'),
]
ax.legend(handles=patches, fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nN. giocatori con |Z| > 2: {(player_stats["z_fg"].abs() > 2).sum()} ',
      f'({(player_stats["z_fg"].abs() > 2).mean()*100:.1f}% — atteso: ~5%)')

**Interpretazione:** Il grafico degli Z-score permette di identificare rapidamente i giocatori eccezionali (Z > 2, rosso) rispetto alla media della lega, indipendentemente dall'unità di misura originale. La percentuale di outlier oltre |Z|=2 dovrebbe avvicinarsi al 5% teorico se la distribuzione fosse normale.

---

## Riepilogo Dispensa 1

| Concetto | Formula chiave | Applicazione NBA |
|---|---|---|
| **Bernoulli** | $P(X=1) = p$ | Prob. di realizzare un tiro |
| **Binomiale PMF** | $\binom{n}{k}p^k(1-p)^{n-k}$ | K canestri su N tiri |
| **CDF** | $F(k) = P(X \leq k)$ | "Al massimo k canestri" |
| **Appross. Normale** | $\mathcal{N}(np,\, np(1-p))$ | Tiri di squadra ($n$ grande) |
| **Appross. Poisson** | $\text{Pois}(\lambda=np)$ | Tiri rari (Alley Oop, ecc.) |
| **Z-score** | $(X - \mu)/\sigma$ | Confronto FG% tra giocatori |

---
*Fine Dispensa 1*

---
# DISPENSA 2
# Legge dei Grandi Numeri, Distribuzione Campionaria, TLC, Inferenza e Stime

---

## 2.1 — Legge dei Grandi Numeri (LGN)

La **Legge dei Grandi Numeri** afferma che, all'aumentare della dimensione campionaria $n$, la media campionaria $\bar{X}_n$ converge in probabilità alla media della popolazione $\mu$:

$$
\bar{X}_n = \frac{1}{n} \sum_{i=1}^{n} X_i \xrightarrow{p} \mu \quad \text{per } n \to \infty
$$

Esistono due versioni:
- **LGN Debole**: $P(|\bar{X}_n - \mu| > \varepsilon) \to 0$ per ogni $\varepsilon > 0$
- **LGN Forte**: $P\!\left(\lim_{n \to \infty} \bar{X}_n = \mu\right) = 1$

**Contesto NBA:** Calcolando la FG% su un numero crescente di tiri, il valore dovrebbe convergere alla vera FG% della popolazione (tutti i tiri NBA mai tentati).

In [ ]:
# ── 2.1 Legge dei Grandi Numeri — convergenza della FG% ──────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import norm, shapiro

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

np.random.seed(42)
sample_lln = df['SHOT_MADE_INT'].sample(50_000, random_state=42).values
cumulative_mean = np.cumsum(sample_lln) / np.arange(1, len(sample_lln) + 1)
true_mean = df['SHOT_MADE_INT'].mean()

print(f'FG% vera (popolazione ~4.45M tiri): {true_mean:.5f}  ({true_mean*100:.3f}%)')
print(f'FG% al termine dei 50k tiri:        {cumulative_mean[-1]:.5f}')
print(f'Differenza finale:                   {abs(true_mean - cumulative_mean[-1]):.6f}')

fig, ax = plt.subplots(figsize=(13, 5))
n_vals = np.arange(1, len(cumulative_mean) + 1)
ax.plot(n_vals, cumulative_mean, color='#3498db', linewidth=1.2,
        label=r'$\bar{X}_n$ (media cumulativa)')
ax.axhline(true_mean, color='red', linestyle='--', linewidth=2,
           label=f'$\\mu$ = {true_mean:.4f}')
ax.fill_between(n_vals, true_mean - 0.01, true_mean + 0.01,
                alpha=0.1, color='red', label='+-1pp dalla media')
ax.set_xscale('log')
ax.set_xlabel('n (numero di tiri, scala logaritmica)')
ax.set_ylabel('FG% cumulativa')
ax.set_title('Legge dei Grandi Numeri — Convergenza della FG% NBA')
ax.legend()
ax.set_ylim(0.35, 0.60)
plt.tight_layout()
plt.show()

**Interpretazione:** Per piccoli $n$ la stima oscilla molto (alta variabilità). All'aumentare di $n$, la curva blu si stabilizza attorno alla retta rossa (vera media della popolazione). Questo dimostra empiricamente la LGN: con abbastanza dati, la media campionaria è un'ottima stima della media della popolazione.

---

## 2.2 — Distribuzione Campionaria della Media

La **distribuzione campionaria** di uno stimatore (es. $\bar{X}$) è la distribuzione di probabilità che lo stimatore assume lavorando su **tutti i possibili campioni** di dimensione $n$ estratti dalla popolazione.

**Proprietà chiave di $\bar{X}$:**
$$
E[\bar{X}] = \mu, \qquad \text{Var}(\bar{X}) = \frac{\sigma^2}{n}, \qquad \text{SE}(\bar{X}) = \frac{\sigma}{\sqrt{n}}
$$

dove SE = **Standard Error** (errore standard), che misura la precisione della stima.

**Contesto NBA:** Estraiamo $B = 5000$ campioni di dimensione $n$ dalla variabile `SHOT_DISTANCE` e osserviamo la distribuzione delle medie campionarie.

In [ ]:
# ── 2.2 Distribuzione Campionaria della Media ─────────────────────────────
distance  = df['SHOT_DISTANCE'].dropna().astype(float)
mu_pop    = distance.mean()
sigma_pop = distance.std(ddof=0)

print(f'Popolazione: mu = {mu_pop:.3f} piedi,  sigma = {sigma_pop:.3f} piedi')

B = 5000
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, n in zip(axes, [10, 50, 200]):
    sample_means = np.array([
        distance.sample(n, random_state=i).mean() for i in range(B)
    ])
    se_teorico  = sigma_pop / np.sqrt(n)
    se_empirico = sample_means.std(ddof=1)
    ax.hist(sample_means, bins=50, density=True,
            color='#3498db', edgecolor='white', linewidth=0.3, alpha=0.85,
            label='Distr. campionaria')
    x_plot = np.linspace(sample_means.min(), sample_means.max(), 300)
    ax.plot(x_plot, norm.pdf(x_plot, mu_pop, se_teorico),
            'r-', linewidth=2.5, label=f'N(mu, sigma/sqrtn)\nSE={se_teorico:.3f}')
    ax.axvline(mu_pop, color='black', linestyle='--', linewidth=1.5,
               label=f'mu={mu_pop:.2f}')
    ax.set_title(f'n = {n}   |   SE empirico = {se_empirico:.3f}')
    ax.set_xlabel('X_bar (media campionaria, piedi)')
    ax.set_ylabel('Densita')
    ax.legend(fontsize=8)

plt.suptitle('Distribuzione Campionaria di X_bar — SHOT_DISTANCE (B=5000 campioni)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nSE empirico vs teorico:')
for n in [10, 50, 200]:
    means = [distance.sample(n, random_state=i).mean() for i in range(1000)]
    print(f'  n={n:3d}: SE empirico={np.std(means):.4f},  SE teorico sigma/sqrt(n)={sigma_pop/np.sqrt(n):.4f}')

**Interpretazione:** All'aumentare di $n$:
1. La distribuzione delle medie campionarie si **stringe** attorno a $\mu$ (il SE diminuisce come $1/\sqrt{n}$)
2. La forma diventa sempre più **campanulare** (normale)
3. I valori empirici del SE coincidono quasi perfettamente con la formula teorica $\sigma/\sqrt{n}$

---

## 2.3 — Teorema del Limite Centrale (TLC)

Il **Teorema del Limite Centrale** è uno dei risultati più importanti della statistica. Afferma che:

> Indipendentemente dalla distribuzione della popolazione, la distribuzione campionaria di $\bar{X}$ si avvicina a una **Normale** al crescere di $n$.

$$
\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} \mathcal{N}(0, 1) \quad \text{per } n \to \infty
$$

Condizioni pratiche:
- Se la popolazione è **simmetrica**: $n \geq 15$ è sufficiente
- Se la popolazione è **asimmetrica**: $n \geq 30$ è la regola empirica

**Contesto NBA:** La variabile `SHOT_MADE` (0/1) ha una distribuzione di Bernoulli — non normale. Il TLC predice che le sue medie campionarie (= FG%) saranno normali per $n$ grande.

In [ ]:
# ── 2.3 Teorema del Limite Centrale ───────────────────────────────────────
pop  = df['SHOT_MADE_INT'].values
mu_p = pop.mean()
sg_p = pop.std(ddof=0)

print(f'Popolazione originale (Bernoulli): mu={mu_p:.4f}, sigma={sg_p:.4f}')
print(f'Skewness popolazione: {pd.Series(pop).skew():.3f}  (0 = simmetrica)')
print(f'Curtosi popolazione:  {pd.Series(pop).kurt():.3f}  (0 = normale)')

B = 4000
n_list = [1, 5, 30, 100]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, n in zip(axes.flatten(), n_list):
    idx_all = np.arange(len(pop))
    np.random.seed(0)
    sample_means = np.array([
        pop[np.random.choice(idx_all, n, replace=False)].mean()
        for _ in range(B)
    ])
    se = sg_p / np.sqrt(n)
    ax.hist(sample_means, bins=50, density=True,
            color='#9b59b6', edgecolor='white', linewidth=0.3, alpha=0.85)
    x_plot = np.linspace(sample_means.min(), sample_means.max(), 300)
    ax.plot(x_plot, norm.pdf(x_plot, mu_p, se), 'r-', linewidth=2.5,
            label=f'N({mu_p:.3f}, {se:.4f}^2)')
    ax.set_title(f'n = {n}', fontsize=13)
    ax.set_xlabel('X_bar (FG% campionaria)')
    ax.set_ylabel('Densita')
    ax.legend(fontsize=9)
    stat, pval = shapiro(sample_means[:500])
    normalita = 'Normale OK' if pval > 0.05 else 'NON normale'
    ax.text(0.05, 0.92, f'Shapiro p={pval:.3f} -> {normalita}',
            transform=ax.transAxes, fontsize=8.5,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('Teorema del Limite Centrale — FG% (var. Bernoulli, B=4000 campioni)', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretazione:** Per $n=1$ vediamo la distribuzione originale di Bernoulli (due picchi: 0 e 1). All'aumentare di $n$, la distribuzione delle medie si avvicina rapidamente alla Normale (curva rossa). Il test di Shapiro-Wilk conferma la normalità già a partire da $n \approx 30$.

---

## 2.4 — Inferenza Statistica: Stime Puntali e Non Distorsione

L'**inferenza statistica** è il processo con cui usiamo un campione per trarre conclusioni sulla popolazione.

**Proprietà desiderabili di uno stimatore $\hat{\theta}$:**

| Proprietà | Definizione | Formula |
|---|---|---|
| **Non distorsione** | In media, centra il parametro vero | $E[\hat{\theta}] = \theta$ |
| **Consistenza** | Converge al vero valore per $n \to \infty$ | $\hat{\theta} \xrightarrow{p} \theta$ |
| **Efficienza** | Minimizza la varianza tra stimatori non distorti | $\text{Var}(\hat{\theta}) \leq \text{Var}(\tilde{\theta})$ |

**Stimatori classici:**
$$
\hat{\mu} = \bar{X} = \frac{1}{n}\sum_{i=1}^n X_i, \qquad S^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})^2
$$

Nota: si usa $n-1$ (non $n$) per rendere $S^2$ **non distorto** per $\sigma^2$.

In [ ]:
# ── 2.4 Non distorsione: S^2 con n-1 vs n ────────────────────────────────
distance   = df['SHOT_DISTANCE'].dropna().astype(float)
sigma2_pop = distance.var(ddof=0)

B = 5000
n = 30
np.random.seed(7)

var_n  = []
var_n1 = []

for _ in range(B):
    samp = distance.sample(n).values
    var_n.append(np.var(samp, ddof=0))
    var_n1.append(np.var(samp, ddof=1))

var_n  = np.array(var_n)
var_n1 = np.array(var_n1)

print(f'Vera varianza della popolazione sigma^2 = {sigma2_pop:.4f}')
print(f'Stimatore 1/n   -> E[S^2_n]   = {var_n.mean():.4f}  | Bias = {var_n.mean()-sigma2_pop:+.4f}  (distorto)')
print(f'Stimatore 1/n-1 -> E[S^2_n-1] = {var_n1.mean():.4f}  | Bias = {var_n1.mean()-sigma2_pop:+.4f}  (non distorto)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (vals, label, color) in zip(axes, [
    (var_n,  'Stimatore 1/n (distorto)',     '#e74c3c'),
    (var_n1, 'Stimatore 1/(n-1) (non dist.)', '#2ecc71')
]):
    ax.hist(vals, bins=50, density=True, color=color,
            edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(vals.mean(), color='blue', linestyle='--', linewidth=2,
               label=f'Media stimatore = {vals.mean():.2f}')
    ax.axvline(sigma2_pop, color='black', linestyle='-', linewidth=2,
               label=f'sigma^2 vera = {sigma2_pop:.2f}')
    ax.set_title(label)
    ax.set_xlabel('Valore stimato di sigma^2')
    ax.set_ylabel('Densita')
    ax.legend(fontsize=9)

plt.suptitle(f'Non distorsione di S^2 (n={n}, B={B}) — SHOT_DISTANCE', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretazione:** Il grafico di sinistra mostra che lo stimatore con $n$ sottostima sistematicamente $\sigma^2$ (la linea blu è a sinistra della nera). Quello con $n-1$ è centrato esattamente sul valore vero: è **non distorto**. Questo spiega perché in statistica si usa sempre $n-1$ al denominatore della varianza campionaria.

---

## 2.5 — Campionamento: Casuale Semplice vs Stratificato

**Metodi di campionamento principali:**
- **Casuale semplice**: ogni elemento ha uguale probabilità di essere selezionato
- **Stratificato**: la popolazione è divisa in strati (es. per posizione) e si campiona proporzionalmente

$$
\text{SE} = \frac{\sigma}{\sqrt{n}} \implies \text{per dimezzare l'errore, serve } 4\times \text{ il campione}
$$

**Contesto NBA:** Confrontiamo le due strategie sulla stima della FG% usando `POSITION_GROUP` come variabile di stratificazione.

In [ ]:
# ── 2.5 Campionamento: semplice vs stratificato ───────────────────────────
pop_valid = df[df['POSITION_GROUP'].isin(['G', 'F', 'C'])].copy()
fg_vera   = pop_valid['SHOT_MADE_INT'].mean()

print(f'FG% vera della popolazione (G/F/C): {fg_vera:.5f}')
print('\nFG% per posizione:')
for pos in ['G', 'F', 'C']:
    fg_pos  = pop_valid[pop_valid['POSITION_GROUP'] == pos]['SHOT_MADE_INT'].mean()
    pct_pop = (pop_valid['POSITION_GROUP'] == pos).mean()
    print(f'  {pos}: FG%={fg_pos:.4f}  |  quota nella popolazione={pct_pop:.3f}')

B     = 2000
n_tot = 300
np.random.seed(42)
pesi  = pop_valid['POSITION_GROUP'].value_counts(normalize=True)

est_semplice     = []
est_stratificato = []

for _ in range(B):
    camp_s = pop_valid.sample(n_tot)
    est_semplice.append(camp_s['SHOT_MADE_INT'].mean())
    strati = []
    for pos in ['G', 'F', 'C']:
        n_strato = max(1, int(round(pesi.get(pos, 0) * n_tot)))
        subset   = pop_valid[pop_valid['POSITION_GROUP'] == pos]
        if len(subset) >= n_strato:
            strati.append(subset.sample(n_strato))
    camp_str = pd.concat(strati)
    est_stratificato.append(camp_str['SHOT_MADE_INT'].mean())

est_semplice     = np.array(est_semplice)
est_stratificato = np.array(est_stratificato)

print(f'\n=== Confronto stimatori (n={n_tot}, B={B}) ===')
print(f'Casuale semplice:   bias={est_semplice.mean()-fg_vera:+.5f}  SE={est_semplice.std():.5f}')
print(f'Stratificato:       bias={est_stratificato.mean()-fg_vera:+.5f}  SE={est_stratificato.std():.5f}')
print(f'Guadagno efficienza: {est_semplice.std()/est_stratificato.std():.3f}x')

fig, ax = plt.subplots(figsize=(11, 5))
bins = np.linspace(
    min(est_semplice.min(), est_stratificato.min()),
    max(est_semplice.max(), est_stratificato.max()), 60)
ax.hist(est_semplice, bins=bins, density=True, alpha=0.6, color='#3498db',
        edgecolor='white', label=f'Casuale semplice  (SE={est_semplice.std():.4f})')
ax.hist(est_stratificato, bins=bins, density=True, alpha=0.6, color='#e74c3c',
        edgecolor='white', label=f'Stratificato       (SE={est_stratificato.std():.4f})')
ax.axvline(fg_vera, color='black', linestyle='--', linewidth=2,
           label=f'FG% vera = {fg_vera:.4f}')
ax.set_xlabel('FG% stimata')
ax.set_ylabel('Densita')
ax.set_title(f'Campionamento Casuale vs Stratificato (n={n_tot}, B={B})')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretazione:** Il campionamento stratificato produce una distribuzione più stretta attorno al vero valore: l'errore standard è minore perché garantisce la rappresentazione proporzionale di Guards, Forwards e Centers, riducendo la variabilità da sotto/sovra-rappresentazione casuale di una posizione.

---

## 2.6 — Rendimenti Decrescenti del Campionamento

$$
\text{SE}(\bar{X}) = \frac{\sigma}{\sqrt{n}} \implies \text{per ridurre SE di un fattore } k\text{, servono } n \cdot k^2 \text{ osservazioni}
$$

Quadruplicare il campione dimezza l'errore — una legge di potenza con esponente $-\frac{1}{2}$.

In [ ]:
# ── 2.6 SE vs dimensione campionaria ──────────────────────────────────────
distance  = df['SHOT_DISTANCE'].dropna().astype(float)
sigma_pop = distance.std(ddof=0)

n_values   = np.array([5, 10, 20, 30, 50, 100, 200, 500, 1000, 2000, 5000])
se_teorico = sigma_pop / np.sqrt(n_values)

B_check     = 500
se_empirico = []
np.random.seed(1)
for n in n_values:
    medie = [distance.sample(n, random_state=i+n).mean() for i in range(B_check)]
    se_empirico.append(np.std(medie, ddof=1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(n_values, se_teorico, 'r-o', linewidth=2.5, markersize=6,
         label='SE teorico = sigma/sqrt(n)')
ax1.plot(n_values, se_empirico, 'b--s', linewidth=1.5, markersize=5,
         alpha=0.8, label='SE empirico')
ax1.set_xlabel('n (dimensione campionaria)')
ax1.set_ylabel('Errore Standard (piedi)')
ax1.set_title('SE vs Dimensione Campionaria')
ax1.legend()

ax2.loglog(n_values, se_teorico, 'r-o', linewidth=2.5, markersize=6, label='SE teorico')
ax2.loglog(n_values, se_empirico, 'b--s', linewidth=1.5, markersize=5,
           alpha=0.8, label='SE empirico')
ax2.set_xlabel('n (scala log)')
ax2.set_ylabel('SE (scala log)')
ax2.set_title('SE vs n — scala log-log\n(pendenza = -1/2)')
ax2.legend()

for n_mark in [50, 200, 1000]:
    idx     = np.where(n_values == n_mark)[0][0]
    se_mark = se_teorico[idx]
    ax1.annotate(f'n={n_mark}\nSE={se_mark:.2f}', xy=(n_mark, se_mark),
                 xytext=(n_mark * 1.4, se_mark + 0.25),
                 arrowprops=dict(arrowstyle='->', color='gray'), fontsize=8)

plt.suptitle('Rendimento Decrescente del Campionamento — SHOT_DISTANCE', fontsize=13)
plt.tight_layout()
plt.show()

print('Dimostrazione: per dimezzare SE serve 4x il campione')
for n_base, n_4x in [(50, 200), (100, 400), (200, 800)]:
    se_base = sigma_pop / np.sqrt(n_base)
    se_4x   = sigma_pop / np.sqrt(n_4x)
    print(f'  SE(n={n_base}) = {se_base:.3f}  ->  SE(n={n_4x}) = {se_4x:.3f}  (rapporto: {se_base/se_4x:.3f}x)')

**Interpretazione:** La curva del SE ha una forma a radice quadrata: i primi campioni aggiuntivi riducono molto l'errore, ma dopo un certo punto il guadagno diventa marginale. In scala log-log la relazione diventa una retta con pendenza $-\frac{1}{2}$, confermando la formula $\text{SE} = \sigma / \sqrt{n}$.

---

## Riepilogo Dispensa 2

| Concetto | Formula chiave | Applicazione NBA |
|---|---|---|
| **LGN** | $\bar{X}_n \xrightarrow{p} \mu$ | FG% cumulativa converge alla vera FG% |
| **Distribuzione campionaria** | $\text{Var}(\bar{X}) = \sigma^2/n$ | Variabilità della media su campioni ripetuti |
| **TLC** | $\frac{\bar{X}-\mu}{\sigma/\sqrt{n}} \to \mathcal{N}(0,1)$ | Medie Bernoulli normali per $n \geq 30$ |
| **Non distorsione** | $E[S^2] = \sigma^2$ con $n-1$ | Varianza campionaria di SHOT_DISTANCE |
| **Campionamento stratificato** | Strati proporzionali | Riduce SE stimando FG% per posizione |
| **Rendimenti decrescenti** | $\text{SE} = \sigma/\sqrt{n}$ | Quadruplicare $n$ dimezza l'errore |

---
*Fine Dispensa 2*

---
# DISPENSA 3
# Distribuzioni t-Student e Chi-Quadrato, Intervalli di Confidenza, Valori Critici

---

## 3.1 — La Distribuzione t di Student

Quando la varianza della popolazione $\sigma^2$ è **ignota** e viene stimata da $S^2$, la statistica:
$$
T = \frac{\bar{X} - \mu}{S / \sqrt{n}} \sim t_{n-1}
$$
segue una distribuzione **t di Student** con $\nu = n-1$ gradi di libertà (df).

**Caratteristiche:**
- Simmetrica attorno a 0, a forma di campana
- Code **più pesanti** della Normale (maggiore incertezza da $\sigma$ ignoto)
- Per $\nu \to \infty$ converge a $\mathcal{N}(0,1)$
- $E[T] = 0$, $\text{Var}(T) = \frac{\nu}{\nu-2}$ per $\nu > 2$

**Regola pratica:** per $n \geq 30$ la $t$ è quasi indistinguibile dalla Normale.

In [ ]:
# ── 3.1 Distribuzione t di Student ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import t as t_dist, chi2, norm

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

x = np.linspace(-5, 5, 500)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PDF per vari df
for nu, color in zip([1, 3, 10, 30], ['#e74c3c','#e67e22','#3498db','#9b59b6']):
    ax1.plot(x, t_dist.pdf(x, nu), color=color, linewidth=2, label=f'df={nu}')
ax1.plot(x, norm.pdf(x), 'k--', linewidth=2.5, label='N(0,1)')
ax1.set_title('PDF t di Student per vari df')
ax1.set_xlabel('t'); ax1.set_ylabel('Densita'); ax1.legend()
ax1.set_ylim(0, 0.45)

# Code più pesanti
ax2.plot(x, t_dist.pdf(x, 3),  '#e74c3c', linewidth=2, label='t(df=3)')
ax2.plot(x, t_dist.pdf(x, 30), '#3498db', linewidth=2, label='t(df=30)')
ax2.plot(x, norm.pdf(x), 'k--', linewidth=2, label='N(0,1)')
ax2.set_xlim(2, 5); ax2.set_ylim(0, 0.04)
ax2.set_title('Code pesanti — zoom per |t| > 2')
ax2.set_xlabel('t'); ax2.set_ylabel('Densita'); ax2.legend()

plt.suptitle('Distribuzione t di Student', fontsize=13)
plt.tight_layout(); plt.show()

print('Valori critici t_{alpha/2} al 95% (IC bilaterale):')
for nu in [5, 10, 20, 30, 60, 120]:
    print(f'  df={nu:3d}: t_0.025 = {t_dist.ppf(0.975, nu):.4f}')
print(f'  z_0.025 (Normale) = {norm.ppf(0.975):.4f}')

**Interpretazione:** Al diminuire dei gradi di libertà, le code della $t$ diventano sempre più pesanti rispetto alla Normale. Questo riflette l'incertezza aggiuntiva introdotta dal dover stimare $\sigma$. Già a $df = 30$ la differenza è trascurabile.

---

## 3.2 — La Distribuzione Chi-Quadrato ($\chi^2$)

Se $Z_1, Z_2, \ldots, Z_k \sim \mathcal{N}(0,1)$ indipendenti, allora:
$$
\chi^2_k = \sum_{i=1}^k Z_i^2 \sim \chi^2(k)
$$
Ha $k$ **gradi di libertà**. È sempre $\geq 0$ e asimmetrica a destra.

**Proprietà:**
$E[\chi^2_k] = k$, $\;\text{Var}(\chi^2_k) = 2k$

**Uso principale:** La statistica campionaria $(n-1)S^2/\sigma^2 \sim \chi^2_{n-1}$, usata per costruire IC sulla varianza e per i test Chi-Quadrato (Dispensa 8).

**Contesto NBA:** Utilizziamo la distribuzione $\chi^2$ per modellare la variabilità della distanza dei tiri in campioni della stagione 2024.

In [ ]:
# ── 3.2 Distribuzione Chi-Quadrato ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_chi = np.linspace(0, 60, 500)
for k, color in zip([2, 5, 10, 20, 30], ['#e74c3c','#e67e22','#2ecc71','#3498db','#9b59b6']):
    ax1.plot(x_chi, chi2.pdf(x_chi, k), color=color, linewidth=2, label=f'k={k}')
ax1.set_title('PDF Chi-Quadrato per vari df'); ax1.set_xlabel('x'); ax1.set_ylabel('Densita')
ax1.legend(); ax1.set_xlim(0, 60); ax1.set_ylim(0, 0.25)

# Verifica empirica: (n-1)*S^2 / sigma^2 ~ chi2(n-1)
dist_2024 = df[df['SEASON_1'] == 2024]['SHOT_DISTANCE'].dropna().astype(float)
sigma2_pop = dist_2024.var(ddof=0)
n_camp = 40
B = 3000
np.random.seed(42)
chi2_obs = [(n_camp - 1) * dist_2024.sample(n_camp).var(ddof=1) / sigma2_pop for _ in range(B)]
chi2_obs = np.array(chi2_obs)

ax2.hist(chi2_obs, bins=60, density=True, color='#3498db', edgecolor='white', linewidth=0.3, alpha=0.85,
         label='Empirico: (n-1)S²/sigma²')
x_th = np.linspace(0, chi2_obs.max() * 1.1, 300)
ax2.plot(x_th, chi2.pdf(x_th, n_camp - 1), 'r-', linewidth=2.5,
         label=f'chi²(df={n_camp-1}) teorica')
ax2.set_title(f'Verifica empirica: (n-1)S²/sigma² ~ chi²({n_camp-1})\n(n={n_camp}, B={B} campioni)')
ax2.set_xlabel('chi²'); ax2.set_ylabel('Densita'); ax2.legend()

plt.suptitle('Distribuzione Chi-Quadrato — SHOT_DISTANCE stagione 2024', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Media empirica chi²: {chi2_obs.mean():.2f}  (attesa: {n_camp-1})')
print(f'Var  empirica chi²:  {chi2_obs.var():.2f}   (attesa: {2*(n_camp-1)})')

**Interpretazione:** La distribuzione empirica di $(n-1)S^2/\sigma^2$ calcolata su migliaia di campioni coincide perfettamente con la curva teorica $\chi^2_{n-1}$, confermando il risultato analitico.

---

## 3.3 — Intervallo di Confidenza per la Media (Z-IC, $\sigma$ noto)

Se $\sigma$ è **noto** e $n$ è grande (o la popolazione è normale), un IC al $(1-\alpha)\%$ è:
$$
\bar{X} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}
$$

dove $z_{\alpha/2}$ è il **valore critico** della Normale standard tale che $P(Z > z_{\alpha/2}) = \alpha/2$.

| Livello confidenza | $\alpha$ | $z_{\alpha/2}$ |
|---|---|---|
| 90% | 0.10 | 1.645 |
| 95% | 0.05 | 1.960 |
| 99% | 0.01 | 2.576 |

**Interpretazione corretta:** Se ripetessimo il campionamento infinite volte, il $(1-\alpha)\%$ degli IC costruiti conterrebbe il vero parametro $\mu$. **Non** significa che $\mu$ è nell'IC con probabilità $1-\alpha$.

In [ ]:
# ── 3.3 Intervallo di Confidenza Z (sigma noto) ───────────────────────────
# Usiamo SHOT_DISTANCE 2024, trattiamo sigma come 'noto' (stimato dalla popolazione)
dist_2024 = df[df['SEASON_1'] == 2024]['SHOT_DISTANCE'].dropna().astype(float)
sigma_known = dist_2024.std(ddof=0)  # sigma 'noto' = deviazione popolazione
mu_true = dist_2024.mean()

n = 100
alpha = 0.05
z_crit = norm.ppf(1 - alpha / 2)

print(f'mu_vera = {mu_true:.4f} piedi')
print(f'sigma   = {sigma_known:.4f} piedi')
print(f'n = {n},  alpha = {alpha},  z_crit = {z_crit:.3f}')

# Simulazione: 100 IC indipendenti
B_ic = 100
np.random.seed(0)
contiene_mu = 0
ic_list = []
for i in range(B_ic):
    samp = dist_2024.sample(n, random_state=i)
    xbar = samp.mean()
    margin = z_crit * sigma_known / np.sqrt(n)
    lo, hi = xbar - margin, xbar + margin
    ic_list.append((lo, xbar, hi))
    if lo <= mu_true <= hi:
        contiene_mu += 1

print(f'\nIC al 95%: {contiene_mu}/{B_ic} contengono mu_vera ({contiene_mu}%)')

fig, ax = plt.subplots(figsize=(6, 11))
for i, (lo, xbar, hi) in enumerate(ic_list):
    color = '#2ecc71' if lo <= mu_true <= hi else '#e74c3c'
    ax.plot([lo, hi], [i, i], color=color, linewidth=1.2)
    ax.plot(xbar, i, 'o', color=color, markersize=2)
ax.axvline(mu_true, color='black', linestyle='--', linewidth=2,
           label=f'mu_vera = {mu_true:.2f}')
ax.set_xlabel('SHOT_DISTANCE (piedi)')
ax.set_ylabel('Campione #')
ax.set_title(f'100 IC al 95% (Z)\n{contiene_mu}% contengono mu')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

# IC su un singolo campione
camp_ex = dist_2024.sample(n, random_state=99)
xbar_ex = camp_ex.mean()
margin_ex = z_crit * sigma_known / np.sqrt(n)
print(f'\nEsempio IC su un campione di n={n}:')
print(f'  X_bar = {xbar_ex:.4f}')
print(f'  Margin = {margin_ex:.4f}')
print(f'  IC 95% = [{xbar_ex-margin_ex:.4f}, {xbar_ex+margin_ex:.4f}]')
print(f'  Ampiezza IC = {2*margin_ex:.4f} piedi')

**Interpretazione:** Le barre verdi contengono il vero parametro $\mu$, quelle rosse no. In media il 95% è verde — esattamente come promesso dalla teoria. Gli IC non sono tutti uguali: la larghezza è fissa (stesso $n$ e $\sigma$), ma il centro $\bar{X}$ varia da campione a campione.

---

## 3.4 — Intervallo di Confidenza per la Media (t-IC, $\sigma$ ignoto)

Nella pratica reale $\sigma$ è quasi sempre **ignoto** e si sostituisce con $S$. La statistica $T = (\bar{X}-\mu)/(S/\sqrt{n}) \sim t_{n-1}$ porta a:
$$
\bar{X} \pm t_{\alpha/2,\, n-1} \cdot \frac{S}{\sqrt{n}}
$$

**Confronto con Z-IC:** Il t-IC è sempre **più largo** (maggiore incertezza su $\sigma$), ma la differenza si riduce all'aumentare di $n$.

**Contesto NBA:** Stimiamo la distanza media dei tiri da 3 punti nella stagione 2024, usando un campione con $\sigma$ ignoto.

In [ ]:
# ── 3.4 Intervallo di Confidenza t (sigma ignoto) ─────────────────────────
dist_3pt = df[(df['SEASON_1']==2024) & (df['SHOT_TYPE']=='3PT Field Goal')]\
             ['SHOT_DISTANCE'].dropna().astype(float)
mu_pop_3pt = dist_3pt.mean()

print(f'Popolazione 3PT: mu = {mu_pop_3pt:.4f} piedi  (n={len(dist_3pt):,})')

risultati = []
for n in [10, 20, 50, 100]:
    np.random.seed(42)
    camp = dist_3pt.sample(n)
    xbar = camp.mean()
    s    = camp.std(ddof=1)
    se   = s / np.sqrt(n)
    t_c  = t_dist.ppf(0.975, df=n-1)
    z_c  = norm.ppf(0.975)
    lo_t = xbar - t_c * se;  hi_t = xbar + t_c * se
    lo_z = xbar - z_c * se;  hi_z = xbar + z_c * se
    risultati.append({'n': n, 'x_bar': xbar, 's': s,
                      't_crit': t_c, 'z_crit': z_c,
                      'ampiezza_t': hi_t-lo_t, 'ampiezza_z': hi_z-lo_z,
                      'lo_t': lo_t, 'hi_t': hi_t})
    print(f'n={n:3d}: x_bar={xbar:.2f}, s={s:.2f}, t={t_c:.3f}, z={z_c:.3f}')
    print(f'       IC-t: [{lo_t:.2f}, {hi_t:.2f}]  ampiezza={hi_t-lo_t:.3f}')
    print(f'       IC-z: [{lo_z:.2f}, {hi_z:.2f}]  ampiezza={hi_z-lo_z:.3f}')
    print()

res_df = pd.DataFrame(risultati)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(res_df['n'], res_df['ampiezza_t'], 'b-o', linewidth=2, markersize=8, label='IC-t (sigma ignoto)')
ax1.plot(res_df['n'], res_df['ampiezza_z'], 'r--s', linewidth=2, markersize=8, label='IC-z (sigma noto)')
ax1.set_xlabel('n'); ax1.set_ylabel('Ampiezza IC (piedi)')
ax1.set_title('Ampiezza IC-t vs IC-z al variare di n')
ax1.legend()

y_pos = range(len(risultati))
for i, r in enumerate(risultati):
    color = '#2ecc71' if r['lo_t'] <= mu_pop_3pt <= r['hi_t'] else '#e74c3c'
    ax2.plot([r['lo_t'], r['hi_t']], [i, i], color=color, linewidth=3)
    ax2.plot(r['x_bar'], i, 'ko', markersize=7)
ax2.axvline(mu_pop_3pt, color='red', linestyle='--', linewidth=2,
            label=f'mu_vera={mu_pop_3pt:.2f}')
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels([f'n={r["n"]}' for r in risultati])
ax2.set_xlabel('Distanza (piedi)')
ax2.set_title('IC-t al 95% per vari n\n(3PT, stagione 2024)')
ax2.legend()

plt.suptitle('Intervalli di Confidenza t — SHOT_DISTANCE 3PT', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Per $n$ piccoli il t-IC è notevolmente più largo del Z-IC (più incertezza su $\sigma$). All'aumentare di $n$ le due ampiezze convergono, perché $t_{n-1} \to z$ per $n \to \infty$. L'IC si stringe anche perché SE $= S/\sqrt{n}$ diminuisce.

---

## 3.5 — Ampiezza dell'IC e Determinazione di $n$

L'ampiezza totale di un IC (Z-IC) è $2 \cdot z_{\alpha/2} \cdot \sigma / \sqrt{n}$.

Se vogliamo che il **margine di errore** (semi-ampiezza) non superi $E$:
$$
n \geq \left( \frac{z_{\alpha/2} \cdot \sigma}{E} \right)^2
$$

**Contesto NBA:** Quanti tiri dobbiamo campionare per stimare la distanza media con un errore massimo di 0.5 piedi al 95%?

In [ ]:
# ── 3.5 Determinazione della dimensione campionaria ───────────────────────
dist_all = df['SHOT_DISTANCE'].dropna().astype(float)
sigma_est = dist_all.std(ddof=1)  # stima di sigma

z95 = norm.ppf(0.975)
z99 = norm.ppf(0.995)

print('Determinazione di n: n >= (z * sigma / E)^2')
print(f'sigma stimata = {sigma_est:.4f} piedi\n')
print(f'{'Errore E':>10} | {'n (95%)':>8} | {'n (99%)':>8}')
print('-' * 34)
E_vals = [0.1, 0.2, 0.5, 1.0, 2.0]
for E in E_vals:
    n95 = int(np.ceil((z95 * sigma_est / E) ** 2))
    n99 = int(np.ceil((z99 * sigma_est / E) ** 2))
    print(f'{E:>10.1f} | {n95:>8,} | {n99:>8,}')

E_range = np.linspace(0.05, 3, 300)
n95_range = np.ceil((z95 * sigma_est / E_range) ** 2)
n99_range = np.ceil((z99 * sigma_est / E_range) ** 2)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(E_range, n95_range, '#3498db', linewidth=2.5, label='IC 95%')
ax.plot(E_range, n99_range, '#e74c3c', linewidth=2.5, label='IC 99%')
ax.axhline(1000, color='gray', linestyle=':', linewidth=1.2, label='n=1000')
ax.axhline(5000, color='gray', linestyle='--', linewidth=1.2, label='n=5000')
ax.set_xlabel('Margine di errore E (piedi)')
ax.set_ylabel('n minimo richiesto')
ax.set_title(f'Dimensione campionaria richiesta — SHOT_DISTANCE (sigma={sigma_est:.2f})')
ax.set_ylim(0, 20000)
ax.legend()
plt.tight_layout(); plt.show()

**Interpretazione:** La curva mostra i rendimenti decrescenti: per ridurre l'errore da 1 piede a 0.5 piedi occorre **quadruplicare** il campione. Passare dal 95% al 99% richiede campioni notevolmente più grandi, specialmente per errori piccoli.

---

## Riepilogo Dispensa 3

| Concetto | Formula chiave | Applicazione NBA |
|---|---|---|
| **t di Student** | $T = (\bar{X}-\mu)/(S/\sqrt{n}) \sim t_{n-1}$ | Code pesanti per $n$ piccoli |
| **Chi-Quadrato** | $(n-1)S^2/\sigma^2 \sim \chi^2_{n-1}$ | Verifica variabilità distanza tiri |
| **IC-Z** (sigma noto) | $\bar{X} \pm z_{\alpha/2} \sigma/\sqrt{n}$ | 95% degli IC costruiti contengono $\mu$ |
| **IC-t** (sigma ignoto) | $\bar{X} \pm t_{\alpha/2,n-1} S/\sqrt{n}$ | IC distanza 3PT per vari $n$ |
| **Ampiezza IC** | $2 z_{\alpha/2} \sigma/\sqrt{n}$ | Cresce al ridursi di $n$ o $\alpha$ |
| **Determinazione n** | $n \geq (z_{\alpha/2}\sigma/E)^2$ | Campione per errore $\leq E$ piedi |

---
*Fine Dispensa 3*

---
# DISPENSA 4
# Proporzione Campionaria e Introduzione ai Test di Ipotesi

---

## 4.1 — La Proporzione Campionaria

Sia $X$ il numero di successi in $n$ prove Bernoulli indipendenti con probabilità $p$. La **proporzione campionaria** è:
$$
\hat{p} = \frac{X}{n}
$$

**Proprietà:**
$$
E[\hat{p}] = p, \qquad \text{Var}(\hat{p}) = \frac{p(1-p)}{n}, \qquad \text{SE}(\hat{p}) = \sqrt{\frac{p(1-p)}{n}}
$$

Per $n$ grande, dal TLC:
$$
\frac{\hat{p} - p}{\sqrt{p(1-p)/n}} \xrightarrow{d} \mathcal{N}(0,1)
$$

**Regola pratica:** l'approssimazione Normale è valida se $np \geq 5$ e $n(1-p) \geq 5$.

**Contesto NBA:** $\hat{p}$ = FG% campionaria. Stimiamo la proporzione di tiri realizzati per posizione (Guards vs Centers) e costruiamo IC per la proporzione.

In [ ]:
# ── 4.1 Proporzione campionaria e distribuzione campionaria di p_hat ──────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, binom

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

# Stima di p per posizione nella stagione 2024
df24 = df[df['SEASON_1'] == 2024]
print('FG% per posizione (stagione 2024):')
for pos in ['G', 'F', 'C']:
    sub = df24[df24['POSITION_GROUP'] == pos]
    p_hat = sub['SHOT_MADE_INT'].mean()
    n_pos = len(sub)
    se = np.sqrt(p_hat * (1 - p_hat) / n_pos)
    print(f'  {pos}: p_hat={p_hat:.4f}, n={n_pos:,}, SE={se:.5f}')

# Distribuzione campionaria di p_hat per vari n
pop_shots = df['SHOT_MADE_INT'].values
p_true = pop_shots.mean()
B = 4000
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, n in zip(axes, [30, 100, 500]):
    np.random.seed(42)
    p_hats = np.array([pop_shots[np.random.choice(len(pop_shots), n)].mean() for _ in range(B)])
    se_th = np.sqrt(p_true * (1 - p_true) / n)
    ax.hist(p_hats, bins=50, density=True, color='#9b59b6',
            edgecolor='white', linewidth=0.3, alpha=0.85)
    x_plot = np.linspace(p_hats.min(), p_hats.max(), 300)
    ax.plot(x_plot, norm.pdf(x_plot, p_true, se_th), 'r-', linewidth=2.5,
            label=f'N(p, p(1-p)/n)\nSE={se_th:.4f}')
    ax.axvline(p_true, color='black', linestyle='--', linewidth=1.5,
               label=f'p={p_true:.3f}')
    ax.set_title(f'n = {n}')
    ax.set_xlabel('p_hat (FG% campionaria)')
    ax.set_ylabel('Densita')
    ax.legend(fontsize=9)
plt.suptitle('Distribuzione campionaria di p_hat (B=4000 campioni)', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** La distribuzione di $\hat{p}$ si avvicina rapidamente alla Normale all'aumentare di $n$, con SE che diminuisce come $\sqrt{p(1-p)/n}$. Per $n=500$ la curva teorica si sovrappone perfettamente all'istogramma empirico.

---

## 4.2 — Intervallo di Confidenza per la Proporzione

L'IC al $(1-\alpha)\%$ per $p$ (metodo di Wald):
$$
\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}
$$

**Nota:** Il metodo di Wald può essere impreciso per $p$ vicino a 0 o 1, o per $n$ piccolo. Esistono alternative (IC di Wilson, Clopper-Pearson) più accurate.

**Determinazione di $n$** per margine $\leq E$:
$$
n \geq \frac{z_{\alpha/2}^2 \cdot p(1-p)}{E^2} \leq \frac{z_{\alpha/2}^2}{4E^2} \quad \text{(caso peggiore } p=0.5\text{)}
$$

In [ ]:
# ── 4.2 IC per la proporzione ──────────────────────────────────────────────
# IC per FG% da 3 punti nella stagione 2024, per vari n
sub_3pt = df[(df['SEASON_1']==2024) & (df['SHOT_TYPE']=='3PT Field Goal')]
p_pop_3pt = sub_3pt['SHOT_MADE_INT'].mean()

z95 = norm.ppf(0.975)
print(f'FG% da 3PT (vera): {p_pop_3pt:.4f}')
print(f'\nIC al 95% per vari n:')
print(f'{'n':>6} | {'p_hat':>6} | {'Lower':>7} | {'Upper':>7} | {'Ampiezza':>9} | Contiene p?')
print('-' * 60)
for n in [30, 50, 100, 200, 500]:
    camp = sub_3pt.sample(n, random_state=42)
    ph = camp['SHOT_MADE_INT'].mean()
    se = np.sqrt(ph * (1 - ph) / n)
    lo, hi = ph - z95 * se, ph + z95 * se
    ok = 'SI' if lo <= p_pop_3pt <= hi else 'NO'
    print(f'{n:>6} | {ph:>6.4f} | {lo:>7.4f} | {hi:>7.4f} | {hi-lo:>9.4f} | {ok}')

# Determinazione n per margine E
print('\nDimensione campionaria per margine E al 95%:')
print(f'(caso peggiore p=0.5, e caso reale p={p_pop_3pt:.3f})')
print(f'{'E':>6} | {'n (p=0.5)':>10} | {'n (p reale)':>12}')
print('-' * 34)
for E in [0.01, 0.02, 0.05, 0.10]:
    n_worst = int(np.ceil(z95**2 * 0.25 / E**2))
    n_real  = int(np.ceil(z95**2 * p_pop_3pt*(1-p_pop_3pt) / E**2))
    print(f'{E:>6.2f} | {n_worst:>10,} | {n_real:>12,}')

# Confronto Wald vs Wilson IC
from scipy.stats import beta as beta_dist
print('\nConfronto Wald vs Wilson (n=30, p_hat osservato):')
camp30 = sub_3pt.sample(30, random_state=1)
ph30   = camp30['SHOT_MADE_INT'].mean()
n30    = 30
# Wald
se_w = np.sqrt(ph30*(1-ph30)/n30)
wald_lo, wald_hi = ph30 - z95*se_w, ph30 + z95*se_w
# Wilson
x30 = int(ph30 * n30)
wil_lo = (x30 + z95**2/2 - z95*np.sqrt(x30*(n30-x30)/n30 + z95**2/4)) / (n30 + z95**2)
wil_hi = (x30 + z95**2/2 + z95*np.sqrt(x30*(n30-x30)/n30 + z95**2/4)) / (n30 + z95**2)
print(f'  p_hat={ph30:.3f}, n={n30}')
print(f'  Wald:   [{wald_lo:.4f}, {wald_hi:.4f}]  ampiezza={wald_hi-wald_lo:.4f}')
print(f'  Wilson: [{wil_lo:.4f}, {wil_hi:.4f}]  ampiezza={wil_hi-wil_lo:.4f}')

**Interpretazione:** L'IC si stringe all'aumentare di $n$. Il metodo di Wilson è preferibile per $n$ piccoli perché rispetta meglio il vincolo $[0,1]$ e ha copertura più vicina al livello nominale.

---

## 4.3 — Introduzione ai Test di Ipotesi

Un **test di ipotesi** è una procedura per decidere, sulla base dei dati, tra due affermazioni sulla popolazione:

- $H_0$: **Ipotesi nulla** — assunzione di partenza (lo status quo, da falsificare)
- $H_1$: **Ipotesi alternativa** — ciò che vogliamo dimostrare

**Struttura del test:**
1. Formulare $H_0$ e $H_1$
2. Scegliere il livello di significatività $\alpha$ (usualmente 0.05 o 0.01)
3. Calcolare la **statistica test** dai dati
4. Calcolare il **p-value** o confrontare con il **valore critico**
5. Prendere la decisione: rifiutare o non rifiutare $H_0$

**Tipi di ipotesi alternativa:**

| Test | $H_0$ | $H_1$ | Regione critica |
|---|---|---|---|
| Bilaterale | $\mu = \mu_0$ | $\mu \neq \mu_0$ | $|Z| > z_{\alpha/2}$ |
| Unilaterale destro | $\mu \leq \mu_0$ | $\mu > \mu_0$ | $Z > z_\alpha$ |
| Unilaterale sinistro | $\mu \geq \mu_0$ | $\mu < \mu_0$ | $Z < -z_\alpha$ |

In [ ]:
# ── 4.3 Formulazione di H0 e H1 con dati NBA ─────────────────────────────
# Domanda: 'La FG% dei Guards nella stagione 2024 e' diversa dalla media storica?'
fg_storica = df['SHOT_MADE_INT'].mean()   # mu_0: FG% storica su tutte le stagioni
guards_24  = df[(df['SEASON_1']==2024) & (df['POSITION_GROUP']=='G')]
n_g = len(guards_24)
p_hat_g = guards_24['SHOT_MADE_INT'].mean()

print('=== Formulazione del test di ipotesi ===')
print(f'Contesto: FG% dei Guards NBA stagione 2023-24 vs media storica')
print(f'')
print(f'H0: p_guards = {fg_storica:.4f}  (la FG% dei Guards = media storica)')
print(f'H1: p_guards != {fg_storica:.4f} (test BILATERALE)')
print(f'')
print(f'Dati campionari:')
print(f'  n       = {n_g:,}')
print(f'  p_hat   = {p_hat_g:.4f}')
print(f'  mu_0    = {fg_storica:.4f}')

# Regione critica grafica
alpha = 0.05
z_crit = norm.ppf(1 - alpha/2)
x_z = np.linspace(-4, 4, 500)
pdf_z = norm.pdf(x_z)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titoli = ['Test Bilaterale\nH1: mu != mu_0',
          'Test Unilaterale Destro\nH1: mu > mu_0',
          'Test Unilaterale Sinistro\nH1: mu < mu_0']
crit_pairs = [
    [(-4, -z_crit), (z_crit, 4)],
    [(z_crit, 4)],
    [(-4, -z_crit)]
]
z_alpha = norm.ppf(1 - alpha)
for ax, titolo, crit in zip(axes, titoli, crit_pairs):
    ax.plot(x_z, pdf_z, 'k-', linewidth=2)
    for (a, b) in crit:
        mask = (x_z >= a) & (x_z <= b)
        ax.fill_between(x_z[mask], pdf_z[mask], alpha=0.5, color='#e74c3c',
                        label='Regione critica')
    ax.fill_between(x_z, pdf_z,
                    where=~np.array([(any(a<=xi<=b for a,b in crit)) for xi in x_z]),
                    alpha=0.15, color='#2ecc71', label='Regione accettazione')
    ax.set_title(titolo); ax.set_xlabel('Z'); ax.set_ylabel('Densita')
    ax.text(0, 0.15, f'alpha={alpha}', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    if ax == axes[0]:
        ax.legend(fontsize=8)
plt.suptitle('Regioni critiche per diversi tipi di test (alpha=0.05)', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** La **regione critica** (rossa) contiene i valori della statistica test per cui rifiutiamo $H_0$. La sua forma dipende dall'ipotesi alternativa:
- **Bilaterale**: due code ($\alpha/2$ per lato)
- **Unilaterale destro**: una coda destra ($\alpha$ tutta da un lato)
- **Unilaterale sinistro**: una coda sinistra

Scegliere il tipo di test sbagliato cambia radicalmente le conclusioni!

---

## 4.4 — Test Z per la Proporzione

Per testare $H_0: p = p_0$ con campione grande, si usa la statistica:
$$
Z = \frac{\hat{p} - p_0}{\sqrt{p_0(1-p_0)/n}}
$$

Si rifiuta $H_0$ al livello $\alpha$ se $|Z| > z_{\alpha/2}$ (test bilaterale).

**Nota importante:** sotto $H_0$ si usa $p_0$ (non $\hat{p}$) al denominatore.

In [ ]:
# ── 4.4 Z-test per la proporzione ─────────────────────────────────────────
# Test: la FG% dei Centers e' > della media storica?
# H0: p_C = p0   H1: p_C > p0  (test unilaterale DESTRO)
p0 = df['SHOT_MADE_INT'].mean()  # media storica = mu_0
centers_24 = df[(df['SEASON_1']==2024) & (df['POSITION_GROUP']=='C')]
n_c   = len(centers_24)
p_hat_c = centers_24['SHOT_MADE_INT'].mean()
alpha = 0.05

# Statistica Z
Z_obs = (p_hat_c - p0) / np.sqrt(p0 * (1 - p0) / n_c)
z_crit_uni = norm.ppf(1 - alpha)  # unilaterale destro
p_value = 1 - norm.cdf(Z_obs)     # P(Z >= Z_obs)

print('=== Z-test per la proporzione (FG% Centers vs media storica) ===')
print(f'H0: p_Centers = {p0:.4f}')
print(f'H1: p_Centers > {p0:.4f}  (test unilaterale destro)')
print(f'')
print(f'n          = {n_c:,}')
print(f'p_hat      = {p_hat_c:.4f}')
print(f'Z_osservato = {Z_obs:.4f}')
print(f'z_critico  = {z_crit_uni:.4f}  (alpha={alpha})')
print(f'p-value    = {p_value:.6f}')
print(f'')
if Z_obs > z_crit_uni:
    print(f'Decisione: Z_obs={Z_obs:.2f} > z_crit={z_crit_uni:.2f} -> RIFIUTO H0')
    print(f'Conclusione: la FG% dei Centers e significativamente > della media storica')
else:
    print(f'Decisione: Z_obs={Z_obs:.2f} <= z_crit={z_crit_uni:.2f} -> NON rifiuto H0')

# Visualizzazione
fig, ax = plt.subplots(figsize=(11, 5))
x_z = np.linspace(-4, 5, 500)
ax.plot(x_z, norm.pdf(x_z), 'k-', linewidth=2, label='N(0,1) sotto H0')
mask_crit = x_z >= z_crit_uni
ax.fill_between(x_z[mask_crit], norm.pdf(x_z[mask_crit]),
                alpha=0.5, color='#e74c3c', label=f'Regione critica (alpha={alpha})')
ax.fill_between(x_z[~mask_crit], norm.pdf(x_z[~mask_crit]),
                alpha=0.12, color='#2ecc71', label='Regione accettazione')
ax.axvline(Z_obs, color='blue', linestyle='-', linewidth=2.5,
           label=f'Z_obs = {Z_obs:.3f}')
ax.axvline(z_crit_uni, color='red', linestyle='--', linewidth=2,
           label=f'z_crit = {z_crit_uni:.3f}')
ax.fill_between(x_z[x_z >= Z_obs], norm.pdf(x_z[x_z >= Z_obs]),
                alpha=0.4, color='blue', label=f'p-value = {p_value:.5f}')
ax.set_xlabel('Z'); ax.set_ylabel('Densita')
ax.set_title('Z-test unilaterale destro — FG% Centers 2024 vs media storica')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Interpretazione:** Il p-value misura la probabilità di osservare un valore di $Z$ almeno così estremo sotto $H_0$. Se p-value $< \alpha$, i dati sono incompatibili con $H_0$ e lo rifiutiamo. L'area blu nel grafico è proprio il p-value.

---

## Riepilogo Dispensa 4

| Concetto | Formula chiave | Applicazione NBA |
|---|---|---|
| **Proporzione campionaria** | $\hat{p} = X/n$, $\text{Var}=p(1-p)/n$ | FG% per posizione |
| **IC per p (Wald)** | $\hat{p} \pm z_{\alpha/2}\sqrt{\hat{p}(1-\hat{p})/n}$ | IC FG% da 3 punti |
| **Determinazione n** | $n \geq z^2/(4E^2)$ | Campione per errore $\leq E$ |
| **H0 e H1** | Status quo vs alternativa | FG% Guards vs media storica |
| **Regione critica** | $|Z| > z_{\alpha/2}$ (bilat.) | Visualizzazione grafica |
| **Z-test proporzione** | $Z = (\hat{p}-p_0)/\sqrt{p_0(1-p_0)/n}$ | FG% Centers vs storica |

---
*Fine Dispensa 4*

---
# DISPENSA 5
# Teoria dei Test: Errori, Potenza, Statistiche Test e Regioni Critiche

---

## 5.1 — Errori di Tipo I e Tipo II

In un test di ipotesi esistono **quattro possibili esiti**:

| | $H_0$ vera | $H_0$ falsa |
|---|---|---|
| **Non rifiuto $H_0$** | Decisione corretta | **Errore Tipo II** ($\beta$) |
| **Rifiuto $H_0$** | **Errore Tipo I** ($\alpha$) | Decisione corretta |

- **Errore Tipo I** ($\alpha$): rifiutare $H_0$ quando è vera (**falso positivo**)  
  $\alpha = P(\text{rifiuto } H_0 \mid H_0 \text{ vera})$

- **Errore Tipo II** ($\beta$): non rifiutare $H_0$ quando è falsa (**falso negativo**)  
  $\beta = P(\text{non rifiuto } H_0 \mid H_0 \text{ falsa})$

- **Potenza del test**: probabilità di rifiutare $H_0$ quando è effettivamente falsa:  
  $\text{Potenza} = 1 - \beta$

**Tradeoff fondamentale:** Diminuire $\alpha$ aumenta $\beta$ (a parità di $n$). L'unico modo per ridurre entrambi è aumentare $n$.

In [ ]:
# ── 5.1 Visualizzazione Errori Tipo I e II ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t as t_dist

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

# Scenario NBA: test bilaterale sulla FG%
# H0: p = 0.455 (media storica)  H1: p != 0.455
# Vera FG% alternativa: p1 = 0.470
p0   = df['SHOT_MADE_INT'].mean()   # ~0.455
p1   = p0 + 0.015                  # vero valore sotto H1
n    = 200
alpha = 0.05

se0 = np.sqrt(p0 * (1 - p0) / n)   # SE sotto H0
se1 = np.sqrt(p1 * (1 - p1) / n)   # SE sotto H1
z_c = norm.ppf(1 - alpha / 2)
lo_crit = p0 - z_c * se0            # limite critico sinistro
hi_crit = p0 + z_c * se0            # limite critico destro

beta = norm.cdf(hi_crit, p1, se1) - norm.cdf(lo_crit, p1, se1)
potenza = 1 - beta

print(f'p0 (H0) = {p0:.4f},  p1 (vera) = {p1:.4f}')
print(f'n={n}, alpha={alpha}')
print(f'Regione critica: p_hat < {lo_crit:.4f} o p_hat > {hi_crit:.4f}')
print(f'Errore Tipo I  alpha = {alpha:.4f}')
print(f'Errore Tipo II beta  = {beta:.4f}')
print(f'Potenza              = {potenza:.4f}  ({potenza*100:.1f}%)')

x = np.linspace(p0 - 4*se0, p1 + 4*se1, 600)
fig, ax = plt.subplots(figsize=(13, 6))

# Distribuzione sotto H0
y0 = norm.pdf(x, p0, se0)
ax.plot(x, y0, '#3498db', linewidth=2.5, label=f'Distr. sotto H0: N({p0:.3f}, {se0:.4f}^2)')
# Errore tipo I (alpha): code fuori dai limiti critici SOTTO H0
m_lo = x <= lo_crit; m_hi = x >= hi_crit
ax.fill_between(x[m_lo], y0[m_lo], color='#e74c3c', alpha=0.5, label=f'Tipo I (alpha={alpha})')
ax.fill_between(x[m_hi], y0[m_hi], color='#e74c3c', alpha=0.5)

# Distribuzione sotto H1
y1 = norm.pdf(x, p1, se1)
ax.plot(x, y1, '#2ecc71', linewidth=2.5, label=f'Distr. sotto H1: N({p1:.3f}, {se1:.4f}^2)')
# Errore tipo II (beta): area di H1 DENTRO i limiti critici
m_b = (x >= lo_crit) & (x <= hi_crit)
ax.fill_between(x[m_b], y1[m_b], color='#f39c12', alpha=0.5,
                label=f'Tipo II (beta={beta:.3f})')
# Potenza: area di H1 FUORI dai limiti critici
ax.fill_between(x[m_lo], y1[m_lo], color='#9b59b6', alpha=0.5,
                label=f'Potenza={potenza:.3f}')
ax.fill_between(x[m_hi], y1[m_hi], color='#9b59b6', alpha=0.5)

ax.axvline(lo_crit, color='red', linestyle='--', linewidth=1.5)
ax.axvline(hi_crit, color='red', linestyle='--', linewidth=1.5, label='Limiti critici')
ax.set_xlabel('p_hat (FG% campionaria)')
ax.set_ylabel('Densita')
ax.set_title(f'Errori Tipo I, Tipo II e Potenza — FG% NBA (n={n})')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Interpretazione:** Le due curve mostrano la distribuzione di $\hat{p}$ rispettivamente sotto $H_0$ (blu) e sotto il vero valore $H_1$ (verde). L'area arancione (Errore Tipo II) è la sovrapposizione tra le due distribuzioni entro la regione di accettazione. Più le distribuzioni si sovrappongono, più il test ha difficoltà a distinguerle.

---

## 5.2 — Curva di Potenza

La **potenza** dipende da:
- $|\mu_1 - \mu_0|$: dimensione dell'effetto (più grande = più potenza)
- $n$: dimensione campionaria (più grande = più potenza)
- $\alpha$: livello di significatività (più grande = più potenza, ma più Errori Tipo I)
- $\sigma$: variabilità (più piccola = più potenza)

La **curva di potenza** mostra come varia la potenza al variare del vero parametro.

In [ ]:
# ── 5.2 Curva di Potenza ──────────────────────────────────────────────────
p0 = df['SHOT_MADE_INT'].mean()
alpha = 0.05
z_c = norm.ppf(1 - alpha / 2)

p1_vals = np.linspace(p0 - 0.06, p0 + 0.06, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva di potenza per vari n
for n, color in zip([50, 100, 200, 500, 1000], ['#e74c3c','#e67e22','#3498db','#2ecc71','#9b59b6']):
    se0 = np.sqrt(p0 * (1 - p0) / n)
    lo_c = p0 - z_c * se0; hi_c = p0 + z_c * se0
    potenza_vals = []
    for p1 in p1_vals:
        se1 = np.sqrt(p1 * (1 - p1) / n)
        beta = norm.cdf(hi_c, p1, se1) - norm.cdf(lo_c, p1, se1)
        potenza_vals.append(1 - beta)
    axes[0].plot(p1_vals - p0, potenza_vals, color=color, linewidth=2, label=f'n={n}')

axes[0].axhline(0.80, color='gray', linestyle='--', linewidth=1.5, label='Potenza=0.80')
axes[0].axhline(alpha, color='black', linestyle=':', linewidth=1, label=f'alpha={alpha}')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('delta = p1 - p0 (dimensione effetto)')
axes[0].set_ylabel('Potenza (1 - beta)')
axes[0].set_title('Curva di potenza per vari n\n(test bilaterale, alpha=0.05)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)

# Potenza vs n per effetto fisso
n_vals_pot = np.arange(20, 2001, 10)
for delta, color in zip([0.01, 0.02, 0.05], ['#e74c3c','#3498db','#2ecc71']):
    p1_fix = p0 + delta
    potenze = []
    for n in n_vals_pot:
        se0 = np.sqrt(p0 * (1 - p0) / n)
        se1 = np.sqrt(p1_fix * (1 - p1_fix) / n)
        lo_c = p0 - z_c * se0; hi_c = p0 + z_c * se0
        beta = norm.cdf(hi_c, p1_fix, se1) - norm.cdf(lo_c, p1_fix, se1)
        potenze.append(1 - beta)
    axes[1].plot(n_vals_pot, potenze, color=color, linewidth=2, label=f'delta={delta}')

axes[1].axhline(0.80, color='gray', linestyle='--', linewidth=1.5, label='Potenza=0.80')
axes[1].set_xlabel('n (dimensione campionaria)')
axes[1].set_ylabel('Potenza (1 - beta)')
axes[1].set_title('Potenza vs n per varie dimensioni di effetto')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.05)

plt.suptitle('Analisi della Potenza — FG% NBA', fontsize=13)
plt.tight_layout(); plt.show()

# n minimo per potenza = 80%
from scipy.stats import norm as norm_dist
print('n minimo per potenza >= 80% (alpha=0.05, test bilaterale):')
for delta in [0.01, 0.02, 0.05, 0.10]:
    p1_fix = p0 + delta
    for n in range(10, 10001):
        se0 = np.sqrt(p0*(1-p0)/n); se1 = np.sqrt(p1_fix*(1-p1_fix)/n)
        lo_c = p0-z_c*se0; hi_c = p0+z_c*se0
        beta = norm.cdf(hi_c,p1_fix,se1)-norm.cdf(lo_c,p1_fix,se1)
        if 1-beta >= 0.80:
            print(f'  delta={delta:.2f}: n_min = {n}')
            break

**Interpretazione:** La curva di potenza ha forma a 'U rovesciata' — è minima sotto $H_0$ (dove vale $\alpha$) e cresce all'allontanarsi dal valore nullo. Effetti più grandi ($\delta$ grande) o campioni più numerosi aumentano la potenza. La linea tratteggiata al 80% è lo standard accettabile in molti contesti applicativi.

---

## 5.3 — La Statistica Test Z e T

**Statistica Z** (varianza nota o $n$ grande):
$$Z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}} \sim \mathcal{N}(0,1) \text{ sotto } H_0$$

**Statistica T** (varianza ignota, $n$ piccolo):
$$T = \frac{\bar{X} - \mu_0}{S/\sqrt{n}} \sim t_{n-1} \text{ sotto } H_0$$

**Quando usare Z vs T?**
- Usa **Z**: $\sigma$ noto, oppure $n \geq 30$ (approssimazione accettabile)
- Usa **T**: $\sigma$ ignoto e $n < 30$

**Contesto NBA:** Testiamo se la distanza media dei tiri nella stagione 2025 differisce dalla media storica.

In [ ]:
# ── 5.3 Confronto Z-test vs T-test ────────────────────────────────────────
dist_storica = df['SHOT_DISTANCE'].dropna().astype(float)
mu0 = dist_storica.mean()       # media storica = mu_0
sigma_nota = dist_storica.std(ddof=0)  # sigma 'nota'

# Campione dalla stagione 2025
dist_2025 = df[df['SEASON_1']==2025]['SHOT_DISTANCE'].dropna().astype(float)
alpha = 0.05

print(f'mu_0 (media storica): {mu0:.4f} piedi')
print(f'sigma (nota):         {sigma_nota:.4f} piedi')
print(f'Popolazione 2025:     n={len(dist_2025):,}, media={dist_2025.mean():.4f}\n')

for n, label in [(15,'piccolo (n=15)'), (50,'medio (n=50)'), (200,'grande (n=200)')]:
    camp = dist_2025.sample(n, random_state=42)
    xbar = camp.mean(); s = camp.std(ddof=1)

    # Z-test (sigma noto)
    Z_obs = (xbar - mu0) / (sigma_nota / np.sqrt(n))
    pval_z = 2 * (1 - norm.cdf(abs(Z_obs)))
    dec_z = 'RIFIUTO H0' if abs(Z_obs) > norm.ppf(0.975) else 'non rifiuto H0'

    # T-test (sigma ignoto)
    T_obs = (xbar - mu0) / (s / np.sqrt(n))
    pval_t = 2 * (1 - t_dist.cdf(abs(T_obs), df=n-1))
    dec_t = 'RIFIUTO H0' if abs(T_obs) > t_dist.ppf(0.975, df=n-1) else 'non rifiuto H0'

    print(f'--- Campione {label} ---')
    print(f'  x_bar={xbar:.3f}, s={s:.3f}')
    print(f'  Z-test: Z={Z_obs:.3f}, p={pval_z:.4f} -> {dec_z}')
    print(f'  T-test: T={T_obs:.3f}, p={pval_t:.4f} -> {dec_t}')
    print()

# Visualizzazione delle due code
camp_ex = dist_2025.sample(50, random_state=42)
xbar_ex = camp_ex.mean(); s_ex = camp_ex.std(ddof=1)
Z_ex = (xbar_ex - mu0) / (sigma_nota / np.sqrt(50))
T_ex = (xbar_ex - mu0) / (s_ex / np.sqrt(50))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x_range = np.linspace(-4.5, 4.5, 500)
for ax, stat_obs, dist_fn, df_val, label, color in [
    (ax1, Z_ex, norm, None, 'Z-test (sigma noto)', '#3498db'),
    (ax2, T_ex, t_dist, 49, 'T-test (sigma ignoto)', '#e74c3c')
]:
    if df_val:
        y = t_dist.pdf(x_range, df_val)
        pval = 2*(1-t_dist.cdf(abs(stat_obs), df_val))
    else:
        y = norm.pdf(x_range)
        pval = 2*(1-norm.cdf(abs(stat_obs)))
    ax.plot(x_range, y, color=color, linewidth=2.5, label=label)
    mask = abs(x_range) >= abs(stat_obs)
    ax.fill_between(x_range[mask], y[mask], alpha=0.4, color=color,
                    label=f'p-value={pval:.4f}')
    ax.axvline(stat_obs, color='black', linestyle='--', linewidth=2,
               label=f'stat_obs={stat_obs:.3f}')
    ax.axvline(-stat_obs, color='black', linestyle='--', linewidth=2)
    ax.set_xlabel('Statistica test'); ax.set_ylabel('Densita')
    ax.set_title(f'{label}\nn=50, p-value={pval:.4f}')
    ax.legend(fontsize=9)
plt.suptitle('Z-test vs T-test — Distanza tiri 2025 vs media storica', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Con $n=50$, Z-test e T-test danno risultati quasi identici perché $t_{49} \approx \mathcal{N}(0,1)$. Per $n=15$, il T-test ha code più pesanti e dunque un p-value leggermente più alto (è più conservativo), il che è appropriato data la maggiore incertezza su $\sigma$.

---

## 5.4 — Test Unilaterale vs Bilaterale: confronto pratico

La scelta tra test unilaterale e bilaterale **deve essere fatta PRIMA di vedere i dati**, basandosi sulla domanda di ricerca:

- **Bilaterale**: 'È cambiato qualcosa?' → $H_1: \mu \neq \mu_0$
- **Unilaterale**: 'È aumentato/diminuito?' → $H_1: \mu > \mu_0$ oppure $\mu < \mu_0$

**Attenzione:** Il test unilaterale ha più potenza nella direzione ipotizzata, ma se l'effetto è nella direzione opposta non lo rileva.

In [ ]:
# ── 5.4 Confronto test unilaterale vs bilaterale ──────────────────────────
# Scenario: distanza media tiri da 3PT aumentata nella stagione 2025 rispetto al 2024?
dist_24 = df[(df['SEASON_1']==2024)&(df['SHOT_TYPE']=='3PT Field Goal')]\
           ['SHOT_DISTANCE'].dropna().astype(float)
dist_25 = df[(df['SEASON_1']==2025)&(df['SHOT_TYPE']=='3PT Field Goal')]\
           ['SHOT_DISTANCE'].dropna().astype(float)

mu0_3pt = dist_24.mean()  # media 2024 come riferimento H0
sigma_3pt = dist_24.std(ddof=0)
alpha = 0.05

print(f'Distanza media 3PT 2024 (mu0): {mu0_3pt:.4f} piedi')
print(f'Distanza media 3PT 2025:       {dist_25.mean():.4f} piedi')
print(f'sigma (dal 2024):              {sigma_3pt:.4f}\n')

risultati = []
for n in [30, 60, 100, 200]:
    camp = dist_25.sample(n, random_state=42)
    xbar = camp.mean(); s = camp.std(ddof=1)
    T_obs = (xbar - mu0_3pt) / (s / np.sqrt(n))
    t_c_bi  = t_dist.ppf(0.975, df=n-1)
    t_c_uni = t_dist.ppf(0.95,  df=n-1)
    pval_bi  = 2*(1-t_dist.cdf(abs(T_obs), df=n-1))
    pval_uni = 1-t_dist.cdf(T_obs, df=n-1)  # unilaterale destro
    r_bi  = 'RIFIUTO' if abs(T_obs) > t_c_bi  else 'non rif.'
    r_uni = 'RIFIUTO' if T_obs      > t_c_uni else 'non rif.'
    risultati.append({'n':n,'T':T_obs,'pv_bi':pval_bi,'pv_uni':pval_uni,
                      'r_bi':r_bi,'r_uni':r_uni})
    print(f'n={n:3d}: T={T_obs:.3f} | Bilat p={pval_bi:.4f} ({r_bi}) | Unilat p={pval_uni:.4f} ({r_uni})')

# Grafico: p-value bilaterale vs unilaterale
ns = [r['n'] for r in risultati]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, [r['pv_bi'] for r in risultati], 'b-o', linewidth=2, markersize=8,
        label='p-value bilaterale')
ax.plot(ns, [r['pv_uni'] for r in risultati], 'g-s', linewidth=2, markersize=8,
        label='p-value unilaterale (destro)')
ax.axhline(alpha, color='red', linestyle='--', linewidth=2, label=f'alpha={alpha}')
ax.set_xlabel('n'); ax.set_ylabel('p-value')
ax.set_title('p-value: test bilaterale vs unilaterale\n(distanza 3PT 2025 vs 2024)')
ax.legend(); plt.tight_layout(); plt.show()

print('\nOsservazione: il p-value unilaterale e sempre la meta del bilaterale')
print('(quando la statistica e nella direzione attesa)')

**Interpretazione:** Il p-value del test unilaterale è sempre la metà di quello bilaterale quando la statistica va nella direzione attesa. Questo rende il test unilaterale più sensibile, ma è scorretto usarlo post-hoc dopo aver visto la direzione dei dati.

---

## Riepilogo Dispensa 5

| Concetto | Formula | Applicazione NBA |
|---|---|---|
| **Errore Tipo I** | $\alpha = P(\text{rif.}H_0 \mid H_0\text{ vera})$ | Soglia di significatività |
| **Errore Tipo II** | $\beta = P(\text{non rif.}H_0 \mid H_0\text{ falsa})$ | Area sovrapposta distribuzioni |
| **Potenza** | $1-\beta$ | Probabilità di rilevare effetto reale |
| **Curva di potenza** | Funzione di $\delta$, $n$, $\alpha$, $\sigma$ | Effetto FG% NBA |
| **Z-test** | $Z=(\bar{X}-\mu_0)/(\sigma/\sqrt{n})$ | $\sigma$ noto o $n\geq 30$ |
| **T-test** | $T=(\bar{X}-\mu_0)/(S/\sqrt{n})$ | $\sigma$ ignoto, $n$ piccolo |
| **Unilat. vs Bilat.** | p-value unilat. = p-value bilat./2 | Scegliere prima di vedere i dati |

---
*Fine Dispensa 5*

---
# DISPENSA 6
# Esercizi Pratici: Intervalli di Confidenza, Z-test, T-test, Errore Tipo II

---

## 6.1 — Esercizio 1: IC per la Media (Z-IC e T-IC a confronto)

**Problema:** Un analista vuole stimare la distanza media dei tiri tentati dai Guards NBA nella stagione 2024. Dispone di un campione di $n = 45$ tiri. Costruire un IC al 90%, 95% e 99% con entrambi i metodi Z e t.

**Procedura:**
1. Calcolare $\bar{X}$ e $S$ dal campione
2. IC-Z: $\bar{X} \pm z_{\alpha/2} \cdot \sigma / \sqrt{n}$ (usando $\sigma$ storico)
3. IC-t: $\bar{X} \pm t_{\alpha/2, n-1} \cdot S / \sqrt{n}$
4. Interpretare e confrontare

In [ ]:
# ── 6.1 IC per la media — esercizio guidato ───────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t as t_dist

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

# Campione
guards_24 = df[(df['SEASON_1']==2024) & (df['POSITION_GROUP']=='G')]
np.random.seed(7)
n = 45
camp = guards_24['SHOT_DISTANCE'].dropna().astype(float).sample(n, random_state=7)
sigma_storico = df['SHOT_DISTANCE'].dropna().astype(float).std(ddof=0)  # sigma noto

xbar = camp.mean()
s    = camp.std(ddof=1)
se_z = sigma_storico / np.sqrt(n)
se_t = s / np.sqrt(n)

print('=== Statistiche campionarie ===')
print(f'n     = {n}')
print(f'X_bar = {xbar:.4f} piedi')
print(f'S     = {s:.4f} piedi')
print(f'sigma (storico) = {sigma_storico:.4f} piedi')
print(f'SE_z = {se_z:.4f},  SE_t = {se_t:.4f}')

print('\n=== Intervalli di Confidenza ===')
print(f'{'Livello':>8} | {'Z-crit':>7} | {'t-crit':>7} | {'IC-Z':>22} | {'IC-t':>22} | Ampiezza Z | Ampiezza t')
print('-'*100)
for conf, alpha in [(0.90, 0.10), (0.95, 0.05), (0.99, 0.01)]:
    z_c = norm.ppf(1 - alpha/2)
    t_c = t_dist.ppf(1 - alpha/2, df=n-1)
    lo_z, hi_z = xbar - z_c*se_z, xbar + z_c*se_z
    lo_t, hi_t = xbar - t_c*se_t, xbar + t_c*se_t
    print(f'{conf*100:.0f}%  |  {z_c:.4f} |  {t_c:.4f} | [{lo_z:.3f}, {hi_z:.3f}] | [{lo_t:.3f}, {hi_t:.3f}] | '
          f'{hi_z-lo_z:.4f}     | {hi_t-lo_t:.4f}')

# Visualizzazione comparativa
fig, ax = plt.subplots(figsize=(11, 5))
livelli = [0.90, 0.95, 0.99]
alphas  = [0.10, 0.05, 0.01]
y_pos   = np.arange(len(livelli))
for i, (conf, alpha) in enumerate(zip(livelli, alphas)):
    z_c = norm.ppf(1 - alpha/2)
    t_c = t_dist.ppf(1 - alpha/2, df=n-1)
    lo_z, hi_z = xbar - z_c*se_z, xbar + z_c*se_z
    lo_t, hi_t = xbar - t_c*se_t, xbar + t_c*se_t
    ax.plot([lo_z, hi_z], [i+0.15, i+0.15], 'b-', linewidth=4, alpha=0.7,
            label='IC-Z' if i==0 else '')
    ax.plot([lo_t, hi_t], [i-0.15, i-0.15], 'r-', linewidth=4, alpha=0.7,
            label='IC-t' if i==0 else '')
    ax.plot(xbar, i+0.15, 'bo', markersize=8)
    ax.plot(xbar, i-0.15, 'ro', markersize=8)
ax.set_yticks(y_pos); ax.set_yticklabels([f'{c*100:.0f}%' for c in livelli])
ax.set_xlabel('Distanza (piedi)')
ax.set_title(f'IC-Z vs IC-t per vari livelli — Guards NBA 2024 (n={n})')
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

**Soluzione e interpretazione:** L'IC-t è leggermente più largo dell'IC-Z perché usa i valori critici della $t_{44}$ invece della Normale. All'aumentare del livello di confidenza, entrambi gli IC si allargano. Con $n=45$, la differenza tra Z e t è piccola ma non trascurabile al 99%.

---

## 6.2 — Esercizio 2: Z-test su media

**Problema:** Si afferma che i tiri da 3 punti nell'era moderna (dal 2015 in poi) vengono tentati da una distanza media maggiore di 25 piedi. Testare questa ipotesi con un campione di $n = 200$ tiri.

**Passi:**
- $H_0: \mu_{3PT} \leq 25$ piedi
- $H_1: \mu_{3PT} > 25$ piedi (test unilaterale destro)
- $\alpha = 0.01$
- Calcolare Z, valore critico, p-value, decisione

In [ ]:
# ── 6.2 Z-test sulla media — soluzione guidata ────────────────────────────
dist_3pt_modern = df[(df['SEASON_1']>=2015)&(df['SHOT_TYPE']=='3PT Field Goal')]\
                   ['SHOT_DISTANCE'].dropna().astype(float)
sigma_3pt = dist_3pt_modern.std(ddof=0)  # sigma noto
mu0 = 25.0
alpha = 0.01
n = 200
np.random.seed(13)
camp = dist_3pt_modern.sample(n, random_state=13)
xbar = camp.mean()

Z_obs = (xbar - mu0) / (sigma_3pt / np.sqrt(n))
z_crit = norm.ppf(1 - alpha)  # unilaterale destro
p_val = 1 - norm.cdf(Z_obs)

print('=== Z-test: distanza media tiri 3PT moderna > 25 piedi? ===')
print(f'H0: mu <= {mu0}  |  H1: mu > {mu0}  (unilaterale destro, alpha={alpha})')
print(f'')
print(f'Sigma (popolazione): {sigma_3pt:.4f} piedi')
print(f'n = {n}')
print(f'X_bar = {xbar:.4f} piedi')
print(f'SE = sigma/sqrt(n) = {sigma_3pt/np.sqrt(n):.4f}')
print(f'')
print(f'Z_osservato = (X_bar - mu0) / SE = ({xbar:.4f} - {mu0}) / {sigma_3pt/np.sqrt(n):.4f} = {Z_obs:.4f}')
print(f'z_critico   = z_(1-alpha) = z_{1-alpha:.2f} = {z_crit:.4f}')
print(f'p-value     = P(Z >= Z_obs) = {p_val:.6f}')
print(f'')
if Z_obs > z_crit:
    print(f'Z_obs={Z_obs:.3f} > z_crit={z_crit:.3f} -> RIFIUTO H0 al livello alpha={alpha}')
    print(f'Conclusione: c e evidenza significativa che la distanza media 3PT > 25 piedi')
else:
    print(f'Z_obs={Z_obs:.3f} <= z_crit={z_crit:.3f} -> non rifiuto H0')

# Grafico
x_range = np.linspace(-3.5, max(4, Z_obs+1), 400)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x_range, norm.pdf(x_range), 'k-', linewidth=2)
mask_pval = x_range >= Z_obs
mask_crit = x_range >= z_crit
ax.fill_between(x_range[mask_crit], norm.pdf(x_range[mask_crit]),
                alpha=0.4, color='#e74c3c', label=f'Regione critica (alpha={alpha})')
ax.fill_between(x_range[mask_pval & ~mask_crit], norm.pdf(x_range[mask_pval & ~mask_crit]),
                alpha=0.4, color='#3498db', label=f'p-value addizionale={p_val:.5f}')
ax.axvline(Z_obs,  color='blue', linewidth=2.5, label=f'Z_obs={Z_obs:.3f}')
ax.axvline(z_crit, color='red',  linewidth=2, linestyle='--', label=f'z_crit={z_crit:.3f}')
ax.set_xlabel('Z'); ax.set_ylabel('Densita')
ax.set_title(f'Z-test unilaterale destro — 3PT distance > {mu0}? (n={n}, alpha={alpha})')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Soluzione:** Con un campione di $n=200$ tiri da 3 punti dell'era moderna, possiamo (o non possiamo, a seconda dei dati reali) rifiutare $H_0$ al livello $\alpha=0.01$. Il p-value ci dice quanto sono improbabili i dati osservati se $H_0$ fosse vera.

---

## 6.3 — Esercizio 3: T-test con campione piccolo

**Problema:** Dalla stagione 2025, si estrae un campione di $n = 18$ tiri di un singolo giocatore. La sua distanza media è cambiata rispetto al valore storico di quel tipo di tiro ($\mu_0 = 24.5$ piedi)? Test bilaterale con $\alpha = 0.05$.

**Nota:** Con $n = 18$, $\sigma$ è ignoto e la distribuzione $t_{17}$ è necessaria.

In [ ]:
# ── 6.3 T-test con campione piccolo — soluzione ───────────────────────────
# Selezioniamo un giocatore con abbastanza tiri nella stagione 2025
df_2025 = df[df['SEASON_1']==2025]
top_players = df_2025.groupby('PLAYER_NAME').size().nlargest(10)
player_name = top_players.index[0]
print(f'Giocatore selezionato: {player_name}')

shots_player = df_2025[df_2025['PLAYER_NAME']==player_name]\
               ['SHOT_DISTANCE'].dropna().astype(float)
mu0 = 24.5
alpha = 0.05
n = 18
np.random.seed(99)
camp = shots_player.sample(min(n, len(shots_player)), random_state=99)
n_eff = len(camp)
xbar = camp.mean(); s = camp.std(ddof=1)
se = s / np.sqrt(n_eff)

T_obs = (xbar - mu0) / se
t_crit = t_dist.ppf(0.975, df=n_eff-1)
p_val = 2 * (1 - t_dist.cdf(abs(T_obs), df=n_eff-1))

print(f'\n=== T-test bilaterale (sigma ignoto, n piccolo) ===')
print(f'H0: mu = {mu0}  |  H1: mu != {mu0}  (bilaterale, alpha={alpha})')
print(f'')
print(f'n     = {n_eff}')
print(f'X_bar = {xbar:.4f} piedi')
print(f'S     = {s:.4f} piedi')
print(f'SE    = S/sqrt(n) = {se:.4f}')
print(f'')
print(f'T_obs   = ({xbar:.4f} - {mu0}) / {se:.4f} = {T_obs:.4f}')
print(f't_crit  = t_0.025,{n_eff-1} = {t_crit:.4f}')
print(f'p-value = 2*P(T_{n_eff-1} >= |T_obs|) = {p_val:.5f}')
print(f'')
if abs(T_obs) > t_crit:
    print(f'|T_obs|={abs(T_obs):.3f} > t_crit={t_crit:.3f} -> RIFIUTO H0')
else:
    print(f'|T_obs|={abs(T_obs):.3f} <= t_crit={t_crit:.3f} -> NON rifiuto H0')

# IC al 95%
lo_ic, hi_ic = xbar - t_crit*se, xbar + t_crit*se
print(f'\nIC al 95%: [{lo_ic:.4f}, {hi_ic:.4f}]')
print(f'mu0={mu0} e nel IC: {lo_ic <= mu0 <= hi_ic}')
print('(Rifiuto H0 <=> mu0 fuori dall IC al 95%: coerente!)')

# Visualizzazione t con dati
x_t = np.linspace(-5, 5, 400)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(x_t, t_dist.pdf(x_t, n_eff-1), 'k-', linewidth=2, label=f't({n_eff-1})')
mask = abs(x_t) >= abs(T_obs)
ax1.fill_between(x_t[mask], t_dist.pdf(x_t[mask], n_eff-1), alpha=0.4, color='#e74c3c',
                 label=f'p-value={p_val:.4f}')
ax1.axvline(T_obs, color='blue', linewidth=2.5, label=f'T_obs={T_obs:.3f}')
ax1.axvline(-T_obs, color='blue', linewidth=2.5)
ax1.axvline(t_crit, color='red', linestyle='--', linewidth=1.5, label=f'+-t_crit={t_crit:.3f}')
ax1.axvline(-t_crit, color='red', linestyle='--', linewidth=1.5)
ax1.set_xlabel('T'); ax1.set_ylabel('Densita')
ax1.set_title(f'T-test bilaterale — {player_name}\n(n={n_eff}, df={n_eff-1})')
ax1.legend(fontsize=9)
ax2.hist(camp.values, bins=15, color='#3498db', edgecolor='white', alpha=0.85, density=True)
ax2.axvline(xbar,  color='blue',  linewidth=2.5, label=f'X_bar={xbar:.2f}')
ax2.axvline(mu0,   color='red',   linewidth=2.5, linestyle='--', label=f'mu0={mu0}')
ax2.axvline(lo_ic, color='green', linewidth=1.5, linestyle=':', label=f'IC 95%')
ax2.axvline(hi_ic, color='green', linewidth=1.5, linestyle=':')
ax2.set_xlabel('Distanza (piedi)'); ax2.set_ylabel('Densita')
ax2.set_title(f'Campione di {n_eff} tiri — {player_name}')
ax2.legend(fontsize=9)
plt.suptitle('T-test con campione piccolo', fontsize=13)
plt.tight_layout(); plt.show()

**Dualità IC–Test:** Se $\mu_0$ è fuori dall'IC al 95%, il test bilaterale a $\alpha=0.05$ rifiuta $H_0$ — e viceversa. Le due procedure sono matematicamente equivalenti.

---

## 6.4 — Esercizio 4: Calcolo dell'Errore Tipo II e della Potenza

**Problema:** Per il test Z unilaterale destro $H_0: \mu \leq 25$ vs $H_1: \mu > 25$ sulla distanza 3PT, calcolare $\beta$ e la potenza per diversi valori veri di $\mu_1$, con $n = 200$ e $\alpha = 0.05$.

$$
\beta = P(\bar{X} \leq c \mid \mu = \mu_1) = \Phi\left(\frac{c - \mu_1}{\sigma/\sqrt{n}}\right)
$$
dove $c = \mu_0 + z_\alpha \cdot \sigma/\sqrt{n}$ è il valore critico campionario.

In [ ]:
# ── 6.4 Calcolo Errore Tipo II e Potenza ──────────────────────────────────
dist_3pt_all = df[df['SHOT_TYPE']=='3PT Field Goal']['SHOT_DISTANCE'].dropna().astype(float)
mu0    = 25.0
sigma  = dist_3pt_all.std(ddof=0)
n      = 200
alpha  = 0.05
z_alpha = norm.ppf(1 - alpha)  # z_0.05 = 1.645
c = mu0 + z_alpha * sigma / np.sqrt(n)  # valore critico campionario

print(f'Test: H0: mu <= {mu0}, H1: mu > {mu0}')
print(f'Parametri: sigma={sigma:.4f}, n={n}, alpha={alpha}')
print(f'z_alpha = {z_alpha:.4f}')
print(f'Valore critico campionario c = mu0 + z*sigma/sqrt(n) = {c:.4f}')
print(f'')
print(f'{'mu_1':>8} | {'delta=mu1-mu0':>13} | {'beta':>8} | {'Potenza':>8}')
print('-'*50)
mu1_vals = [25.0, 25.3, 25.5, 25.8, 26.0, 26.5, 27.0]
for mu1 in mu1_vals:
    beta    = norm.cdf(c, loc=mu1, scale=sigma/np.sqrt(n))
    potenza = 1 - beta
    print(f'{mu1:>8.1f} | {mu1-mu0:>13.1f} | {beta:>8.4f} | {potenza:>8.4f}')

# Grafico beta e potenza vs mu1
mu1_range = np.linspace(mu0, mu0 + 3, 300)
beta_vals    = norm.cdf(c, loc=mu1_range, scale=sigma/np.sqrt(n))
potenza_vals = 1 - beta_vals

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(mu1_range, beta_vals, '#e74c3c', linewidth=2.5, label='Errore Tipo II (beta)')
ax1.axhline(alpha, color='gray', linestyle='--', label=f'alpha={alpha}')
ax1.axvline(mu0, color='black', linestyle=':', linewidth=1)
ax1.set_xlabel('mu_1 (vero valore sotto H1)')
ax1.set_ylabel('beta'); ax1.set_title('Errore Tipo II vs mu_1')
ax1.legend()

ax2.plot(mu1_range, potenza_vals, '#2ecc71', linewidth=2.5, label='Potenza (1-beta)')
ax2.axhline(0.80, color='gray', linestyle='--', label='Potenza=0.80')
ax2.axhline(0.90, color='orange', linestyle=':', label='Potenza=0.90')
mu1_80 = mu1_range[np.argmin(abs(potenza_vals - 0.80))]
ax2.axvline(mu1_80, color='gray', linestyle='--', linewidth=1)
ax2.text(mu1_80+0.05, 0.5, f'mu1={mu1_80:.2f}\nper pot.=80%', fontsize=9)
ax2.set_xlabel('mu_1 (vero valore sotto H1)')
ax2.set_ylabel('Potenza (1-beta)'); ax2.set_title('Curva di Potenza vs mu_1')
ax2.legend()

plt.suptitle(f'Errore Tipo II e Potenza — 3PT distance (n={n}, alpha={alpha})', fontsize=13)
plt.tight_layout(); plt.show()

# n minimo per potenza 80% con delta=0.5 piedi
delta_target = 0.5
mu1_target   = mu0 + delta_target
for n_test in range(10, 5001, 10):
    c_test = mu0 + z_alpha * sigma / np.sqrt(n_test)
    beta_t = norm.cdf(c_test, mu1_target, sigma/np.sqrt(n_test))
    if 1 - beta_t >= 0.80:
        print(f'\nPer delta={delta_target} piedi e potenza>=80%: n_min = {n_test}')
        break

**Interpretazione:** La curva di Errore Tipo II (rossa) decresce al crescere di $\mu_1$: effetti più grandi sono più facili da rilevare. La potenza aumenta specularmente. Il calcolo di $n_{min}$ permette di pianificare uno studio con potenza target del 80%.

---

## Riepilogo Dispensa 6

| Esercizio | Tecnica | Concetto chiave |
|---|---|---|
| **6.1** | IC-Z vs IC-t | Per $n=45$: t-IC leggermente più largo |
| **6.2** | Z-test unilaterale | Distanza 3PT > 25 piedi? |
| **6.3** | T-test bilaterale | $n=18$: usa distribuzione $t_{17}$ |
| **6.4** | Errore Tipo II | $\beta = \Phi((c-\mu_1)/(\sigma/\sqrt{n}))$ |

**Schemi decisionali:**
- $\sigma$ noto $\to$ Z-test / IC-Z
- $\sigma$ ignoto, $n < 30$ $\to$ T-test / IC-t
- $\sigma$ ignoto, $n \geq 30$ $\to$ T-test o Z-test (approx. accettabile)
- Dualità IC–Test: IC al $1-\alpha$ include $\mu_0$ $\Leftrightarrow$ test non rifiuta $H_0$

---
*Fine Dispensa 6*

---
# DISPENSA 7
# P-Value, Test sulla Media Singola e Differenza tra Medie

---

## 7.1 — Il P-Value: definizione rigorosa e interpretazione

Il **p-value** è la probabilità di osservare una statistica test uguale o più estrema di quella osservata, **assumendo che $H_0$ sia vera**:

$$
p\text{-value} = P(|T_{\text{stat}}| \geq |T_{\text{obs}}| \mid H_0)
$$

**Interpretazioni corrette:**
- Un p-value piccolo indica che i dati sono **incompatibili** con $H_0$
- Non è la probabilità che $H_0$ sia vera
- Non è la probabilità di aver fatto un errore

**Regola decisionale:** Rifiuta $H_0$ se p-value $< \alpha$

**Malinteso comune:** p-value $= 0.06$ non significa 'quasi significativo'. La soglia $\alpha$ è fissata **prima** del test.

**Contesto NBA:** Visualiziamo la distribuzione dei p-value sotto $H_0$ vera e sotto $H_1$ alternativa.

In [ ]:
# ── 7.1 Distribuzione del p-value sotto H0 e H1 ──────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t as t_dist, ttest_1samp, ttest_ind

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

dist_all = df['SHOT_DISTANCE'].dropna().astype(float)
mu0_true = dist_all.mean()  # media vera della popolazione
mu1_alt  = mu0_true + 1.5  # valore alternativo sotto H1
n = 50; B = 5000
np.random.seed(42)

# p-value sotto H0 (mu_vero = mu0): uniformi in [0,1]
pvals_h0 = []
for _ in range(B):
    camp = dist_all.sample(n)
    _, pv = ttest_1samp(camp, mu0_true)
    pvals_h0.append(pv)

# p-value sotto H1 (mu_vero = mu1): concentrati vicino a 0
# Costruiamo una popolazione con media mu1
shift = mu1_alt - dist_all.mean()
pop_h1 = dist_all + shift
pvals_h1 = []
for _ in range(B):
    camp = pop_h1.sample(n)
    _, pv = ttest_1samp(camp, mu0_true)  # test ancora su mu0
    pvals_h1.append(pv)

pvals_h0 = np.array(pvals_h0)
pvals_h1 = np.array(pvals_h1)

print(f'Sotto H0 (mu_vera = mu0):   % p<0.05 = {(pvals_h0<0.05).mean()*100:.1f}%  (atteso: 5%)')
print(f'Sotto H1 (mu_vera = mu0+{shift:.1f}): % p<0.05 = {(pvals_h1<0.05).mean()*100:.1f}%  (= potenza)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, pvals, title, color in [
    (ax1, pvals_h0, f'Distribuzione p-value sotto H0 vera\n(uniforme in [0,1])', '#3498db'),
    (ax2, pvals_h1, f'Distribuzione p-value sotto H1 (delta={shift:.1f})\n(concentrata vicino a 0)', '#e74c3c')
]:
    ax.hist(pvals, bins=40, color=color, edgecolor='white', linewidth=0.3, density=True, alpha=0.85)
    ax.axvline(0.05, color='black', linestyle='--', linewidth=2, label='alpha=0.05')
    ax.fill_between([0, 0.05], [0, 0], [ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 5]*2,
                    alpha=0.15, color='red')
    ax.set_xlabel('p-value'); ax.set_ylabel('Densita')
    ax.set_title(title); ax.legend(fontsize=9)
plt.suptitle(f'Distribuzione del p-value (n={n}, B={B} simulazioni)', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione chiave:** Sotto $H_0$ vera, il p-value è **uniformemente distribuito** in $[0,1]$: il 5% dei test risulterà significativo per puro caso ($\alpha$). Sotto $H_1$, il p-value si concentra vicino a 0: il test rileva l'effetto reale (potenza).

---

## 7.2 — Test sulla Media Singola: Z-test e T-test

**Z-test** ($\sigma$ noto): $H_0: \mu = \mu_0$
$$Z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}}, \quad \text{p-value bilat.} = 2\left(1 - \Phi(|Z|)\right)$$

**T-test** ($\sigma$ ignoto): stesso schema con $t_{n-1}$

**Effect size** (Cohen's d): misura la rilevanza pratica dell'effetto, indipendentemente da $n$:
$$d = \frac{\bar{X} - \mu_0}{S}$$
Linee guida: $|d| < 0.2$ piccolo, $|d| < 0.5$ medio, $|d| \geq 0.8$ grande.

**Contesto NBA:** La distanza media dei tiri da 3PT è cambiata tra la stagione 2004 e 2024?

In [ ]:
# ── 7.2 T-test sulla media — distanza 3PT: 2004 vs mu0 ────────────────────
dist_3pt_2004 = df[(df['SEASON_1']==2004)&(df['SHOT_TYPE']=='3PT Field Goal')]\
                 ['SHOT_DISTANCE'].dropna().astype(float)
dist_3pt_2024 = df[(df['SEASON_1']==2024)&(df['SHOT_TYPE']=='3PT Field Goal')]\
                 ['SHOT_DISTANCE'].dropna().astype(float)

mu0_2004 = dist_3pt_2004.mean()  # usiamo 2004 come mu0 di riferimento
print(f'Distanza media 3PT 2004: {mu0_2004:.4f} piedi (usata come mu0)')
print(f'Distanza media 3PT 2024: {dist_3pt_2024.mean():.4f} piedi')
print(f'n disponibili 2024: {len(dist_3pt_2024):,}')

# Campioni di varie dimensioni
alpha = 0.05
print(f'\nTest: H0: mu_2024 = {mu0_2004:.3f}  H1: mu_2024 != {mu0_2004:.3f} (bilaterale)')
print(f'{'n':>5} | {'X_bar':>7} | {'S':>6} | {'T_obs':>7} | {'df':>4} | {'p-value':>10} | {'Cohen d':>8} | Decisione')
print('-'*80)

for n in [20, 50, 100, 300]:
    np.random.seed(42)
    camp = dist_3pt_2024.sample(n, random_state=42)
    xbar = camp.mean(); s = camp.std(ddof=1)
    T_obs, pval = ttest_1samp(camp, mu0_2004)
    d_cohen = (xbar - mu0_2004) / s
    dec = 'RIFIUTO' if pval < alpha else 'non rif.'
    print(f'{n:>5} | {xbar:>7.4f} | {s:>6.3f} | {T_obs:>7.3f} | {n-1:>4} | {pval:>10.6f} | {d_cohen:>8.4f} | {dec}')

# Grafico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ns_list = [10, 20, 30, 50, 70, 100, 150, 200, 300]
T_list = []; pv_list = []; d_list = []
for n in ns_list:
    camp = dist_3pt_2024.sample(n, random_state=42)
    T_obs, pval = ttest_1samp(camp, mu0_2004)
    d = (camp.mean() - mu0_2004) / camp.std(ddof=1)
    T_list.append(T_obs); pv_list.append(pval); d_list.append(d)

axes[0].plot(ns_list, pv_list, 'b-o', linewidth=2, markersize=7)
axes[0].axhline(alpha, color='red', linestyle='--', linewidth=2, label=f'alpha={alpha}')
axes[0].set_xlabel('n'); axes[0].set_ylabel('p-value')
axes[0].set_title('p-value vs dimensione campionaria')
axes[0].legend()

axes[1].plot(ns_list, [abs(d) for d in d_list], 'g-o', linewidth=2, markersize=7)
axes[1].axhline(0.2, color='orange', linestyle=':', linewidth=1.5, label='piccolo (d=0.2)')
axes[1].axhline(0.5, color='orange', linestyle='--', linewidth=1.5, label='medio (d=0.5)')
axes[1].axhline(0.8, color='red',    linestyle='-', linewidth=1.5, label='grande (d=0.8)')
axes[1].set_xlabel('n'); axes[1].set_ylabel('|Cohen d|')
axes[1].set_title('Effect size vs dimensione campionaria')
axes[1].legend(fontsize=9)

plt.suptitle('T-test sulla media — Distanza 3PT 2024 vs 2004', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Il p-value decresce all'aumentare di $n$ (più potenza), mentre il Cohen's $d$ rimane stabile (misura l'effetto, non la significatività). Un test può essere **statisticamente significativo** ma con un effect size piccolo: con $n$ grande, anche differenze trascurabili diventano 'significative'.

---

## 7.3 — Test sulla Differenza tra Due Medie Indipendenti

Per confrontare le medie di **due popolazioni indipendenti** $X \sim N(\mu_1, \sigma_1^2)$ e $Y \sim N(\mu_2, \sigma_2^2)$:

**Caso 1 — Varianze uguali** ($\sigma_1^2 = \sigma_2^2 = \sigma^2$, **pooled**):
$$T = \frac{(\bar{X}_1 - \bar{X}_2) - \Delta_0}{S_p\sqrt{1/n_1 + 1/n_2}} \sim t_{n_1+n_2-2}$$
$$S_p^2 = \frac{(n_1-1)S_1^2 + (n_2-1)S_2^2}{n_1+n_2-2}$$

**Caso 2 — Varianze diverse** (approssimazione di **Welch**):
$$T = \frac{(\bar{X}_1 - \bar{X}_2) - \Delta_0}{\sqrt{S_1^2/n_1 + S_2^2/n_2}} \sim t_{\nu}$$
con gradi di libertà $\nu$ calcolati dalla formula di Welch-Satterthwaite.

**In pratica:** il test di Welch è raccomandato di default (più robusto).

**Contesto NBA:** La distanza media dei tiri da 3 punti è diversa tra Guards e Forwards?

In [ ]:
# ── 7.3 Test differenza tra due medie indipendenti (Welch T-test) ─────────
df24 = df[df['SEASON_1']==2024]
g_dist = df24[(df24['POSITION_GROUP']=='G')&(df24['SHOT_TYPE']=='3PT Field Goal')]\
           ['SHOT_DISTANCE'].dropna().astype(float)
f_dist = df24[(df24['POSITION_GROUP']=='F')&(df24['SHOT_TYPE']=='3PT Field Goal')]\
           ['SHOT_DISTANCE'].dropna().astype(float)

print(f'Guards  3PT: n={len(g_dist):,}, media={g_dist.mean():.4f}, s={g_dist.std(ddof=1):.4f}')
print(f'Forwards 3PT: n={len(f_dist):,}, media={f_dist.mean():.4f}, s={f_dist.std(ddof=1):.4f}')
print(f'Differenza osservata: {g_dist.mean()-f_dist.mean():.4f} piedi')

alpha = 0.05

# Welch T-test (equal_var=False)
T_welch, pval_welch = ttest_ind(g_dist, f_dist, equal_var=False)
# Pooled T-test (equal_var=True)
T_pool, pval_pool   = ttest_ind(g_dist, f_dist, equal_var=True)

# Calcolo manuale Welch
n1, n2 = len(g_dist), len(f_dist)
xbar1, xbar2 = g_dist.mean(), f_dist.mean()
s1, s2 = g_dist.std(ddof=1), f_dist.std(ddof=1)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
T_man = (xbar1 - xbar2) / se_welch
# df Welch-Satterthwaite
nu = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))

print(f'\n=== Test H0: mu_Guards = mu_Forwards  H1: mu_Guards != mu_Forwards ===')
print(f'\nWelch T-test (varianze diverse):')
print(f'  T = {T_welch:.4f},  df_Welch = {nu:.2f},  p-value = {pval_welch:.6f}')
print(f'  Decisione: {"RIFIUTO H0" if pval_welch<alpha else "non rifiuto H0"} (alpha={alpha})')
print(f'\nPooled T-test (varianze uguali):')
print(f'  T = {T_pool:.4f},  p-value = {pval_pool:.6f}')
print(f'  Decisione: {"RIFIUTO H0" if pval_pool<alpha else "non rifiuto H0"} (alpha={alpha})')

# IC per la differenza
t_c = t_dist.ppf(0.975, df=nu)
diff_obs = xbar1 - xbar2
lo_ic = diff_obs - t_c * se_welch
hi_ic = diff_obs + t_c * se_welch
print(f'\nIC al 95% per (mu_G - mu_F): [{lo_ic:.4f}, {hi_ic:.4f}]')
print(f'(0 nel IC: {lo_ic<=0<=hi_ic} -> coerente con test)')

# Cohen d
s_pool = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
d_cohen = diff_obs / s_pool
print(f'\nEffect size Cohen d = {d_cohen:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Distribuzioni a campione di 300 per velocita
g_samp = g_dist.sample(300, random_state=0)
f_samp = f_dist.sample(300, random_state=0)
axes[0].hist(g_samp, bins=30, density=True, alpha=0.6, color='#3498db', label='Guards')
axes[0].hist(f_samp, bins=30, density=True, alpha=0.6, color='#e74c3c', label='Forwards')
axes[0].axvline(g_samp.mean(), color='#3498db', linestyle='--', linewidth=2)
axes[0].axvline(f_samp.mean(), color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_xlabel('Distanza 3PT (piedi)'); axes[0].set_ylabel('Densita')
axes[0].set_title('Distribuzioni di SHOT_DISTANCE 3PT\nGuards vs Forwards (campione 300)')
axes[0].legend()

# IC per la differenza
axes[1].barh(['Differenza\nmu_G - mu_F'], [diff_obs], xerr=[[diff_obs-lo_ic],[hi_ic-diff_obs]],
             color='#9b59b6', alpha=0.8, capsize=8, height=0.4)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='H0: diff=0')
axes[1].set_xlabel('Differenza di distanza (piedi)')
axes[1].set_title(f'IC 95% per (mu_G - mu_F)\n[{lo_ic:.3f}, {hi_ic:.3f}]')
axes[1].legend()
plt.suptitle('Test differenza tra due medie — 3PT Guards vs Forwards 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Se l'IC per la differenza non contiene 0, rifiutiamo $H_0$ — coerente con il test. Welch e Pooled danno risultati simili con campioni grandi, ma Welch è preferibile quando le varianze differiscono (verificare con test di Levene, Dispensa 8).

---

## 7.4 — Test su Dati Appaiati (T-test a Coppie)

Quando le due misurazioni riguardano le **stesse unità** (prima/dopo, stesso giocatore in due stagioni), si usa il **T-test appaiato**:

$$D_i = X_{1i} - X_{2i}, \qquad \bar{D} = \frac{1}{n}\sum D_i, \qquad S_D$$

$$T = \frac{\bar{D} - \Delta_0}{S_D/\sqrt{n}} \sim t_{n-1}$$

**Vantaggio:** Elimina la variabilità tra unità (più potente del test indipendente).

**Contesto NBA:** La FG% di ogni giocatore è cambiata tra la stagione 2022 e 2024? (stessi giocatori, confronto appaiato)

In [ ]:
# ── 7.4 T-test appaiato: FG% giocatori 2022 vs 2024 ──────────────────────
# FG% per giocatore in 2022
fg22 = df[df['SEASON_1']==2022].groupby('PLAYER_NAME')['SHOT_MADE_INT'].agg(['sum','count'])
fg22.columns = ['made22','att22']
fg22['fg22'] = fg22['made22'] / fg22['att22']
fg22 = fg22[fg22['att22'] >= 200]  # almeno 200 tiri per ridurre rumore

fg24 = df[df['SEASON_1']==2024].groupby('PLAYER_NAME')['SHOT_MADE_INT'].agg(['sum','count'])
fg24.columns = ['made24','att24']
fg24['fg24'] = fg24['made24'] / fg24['att24']
fg24 = fg24[fg24['att24'] >= 200]

# Giocatori presenti in entrambe le stagioni
comuni = fg22.join(fg24, how='inner')
n_pair = len(comuni)
print(f'Giocatori con >=200 tiri sia nel 2022 sia nel 2024: {n_pair}')

D = comuni['fg24'] - comuni['fg22']  # differenze appaiata
D_bar = D.mean(); S_D = D.std(ddof=1)
T_paired, pval_paired = ttest_1samp(D, 0)

# T-test indipendente (errato, per confronto)
T_ind, pval_ind = ttest_ind(comuni['fg24'], comuni['fg22'], equal_var=False)

alpha = 0.05
print(f'\nD_bar = {D_bar:.5f}  ({D_bar*100:+.3f}% di FG%)')
print(f'S_D   = {S_D:.5f}')
print(f'n     = {n_pair}')
print(f'\nT-test APPAIATO: T={T_paired:.4f}, p={pval_paired:.5f}')
print(f'  Decisione: {"RIFIUTO H0" if pval_paired<alpha else "non rifiuto H0"} (alpha={alpha})')
print(f'\nT-test INDIPENDENTE (confronto): T={T_ind:.4f}, p={pval_ind:.5f}')
print(f'  Decisione: {"RIFIUTO H0" if pval_ind<alpha else "non rifiuto H0"} (alpha={alpha})')
print(f'\n-> Il test appaiato e piu potente (usa informazione within-player)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(D, bins=30, color='#9b59b6', edgecolor='white', alpha=0.85, density=True)
ax1.axvline(D_bar, color='blue', linewidth=2.5, label=f'D_bar={D_bar:.4f}')
ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='H0: D=0')
# Curva t teorica
x_t = np.linspace(D.min(), D.max(), 300)
from scipy.stats import t as t_dist2
ax1.plot(x_t, t_dist2.pdf((x_t - D_bar)/(S_D/np.sqrt(n_pair)), n_pair-1)/(S_D/np.sqrt(n_pair)),
         'k-', linewidth=2, label=f't({n_pair-1})')
ax1.set_xlabel('Differenza FG% (2024 - 2022)')
ax1.set_ylabel('Densita')
ax1.set_title('Distribuzione delle differenze appaiata\n(FG% 2024 - FG% 2022)')
ax1.legend(fontsize=9)

# Scatter: FG% 2022 vs 2024
ax2.scatter(comuni['fg22'], comuni['fg24'], alpha=0.5, s=30, color='#2ecc71')
lim_min = min(comuni['fg22'].min(), comuni['fg24'].min()) - 0.02
lim_max = max(comuni['fg22'].max(), comuni['fg24'].max()) + 0.02
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=2, label='y=x (no change)')
ax2.set_xlabel('FG% 2022'); ax2.set_ylabel('FG% 2024')
ax2.set_title(f'FG% per giocatore: 2022 vs 2024 (n={n_pair})')
ax2.legend(); ax2.set_xlim(lim_min, lim_max); ax2.set_ylim(lim_min, lim_max)
plt.suptitle('T-test Appaiato — FG% giocatori NBA 2022 vs 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Nel grafico scatter, i punti sopra la diagonale indicano giocatori migliorati, quelli sotto peggiorati. Il T-test appaiato tiene conto di questa struttura: analizzando le **differenze** $D_i = FG\%_{24} - FG\%_{22}$, elimina la variabilità dovuta alle differenze intrinseche tra giocatori (some naturally shoot better).

---

## Riepilogo Dispensa 7

| Concetto | Formula | Applicazione NBA |
|---|---|---|
| **P-value sotto H0** | Uniforme in $[0,1]$ | Simulazione su distanza tiri |
| **P-value sotto H1** | Concentrato vicino a 0 | Rileva effetto reale |
| **T-test 1 campione** | $T=(\bar{X}-\mu_0)/(S/\sqrt{n})$ | Distanza 3PT 2024 vs 2004 |
| **Cohen's d** | $d=(\bar{X}-\mu_0)/S$ | Effect size indip. da $n$ |
| **Welch T-test** | $T_{Welch}$, df Satterthwaite | Guards vs Forwards 3PT |
| **T-test appaiato** | $T=\bar{D}/(S_D/\sqrt{n})$ | FG% per giocatore 2022 vs 2024 |

---
*Fine Dispensa 7*

---
# DISPENSA 8
# Test Avanzati: Normalità, Chi-Quadrato, Omoschedasticità, ANOVA, Correlazione, Regressione

---

## 8.1 — Test di Normalità

Molti test statistici assumono la normalità. Prima di applicarli, è buona pratica verificarla.

**Test principali:**
- **Shapiro-Wilk**: potente per $n < 50$, $H_0$: la distribuzione è normale
- **Kolmogorov-Smirnov**: confronta CDF empirica vs teorica
- **Jarque-Bera**: basato su skewness e curtosi, potente per $n$ grande
- **Q-Q Plot**: strumento grafico indispensabile

**Nota:** Per $n$ molto grande, anche piccole deviazioni dalla normalità risultano significative. Combinare sempre i test grafici con quelli formali.

In [ ]:
# ── 8.1 Test di Normalità ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (norm, shapiro, kstest, jarque_bera,
                          chi2, f as f_dist, pearsonr, spearmanr,
                          levene, bartlett)
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

try:
    df
except NameError:
    df = pd.read_csv('data/shots_all_seasons.csv')
    df['SHOT_MADE'] = df['SHOT_MADE'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
    df['SHOT_MADE_INT'] = df['SHOT_MADE'].astype(int)

# Campione di SHOT_DISTANCE per stagione 2024
np.random.seed(0)
dist_24 = df[df['SEASON_1']==2024]['SHOT_DISTANCE'].dropna().astype(float)
camp_small = dist_24.sample(40, random_state=0)   # n piccolo -> Shapiro
camp_large = dist_24.sample(500, random_state=1)  # n grande

print('=== Test di Normalita — SHOT_DISTANCE 2024 ===')
for label, samp in [('n=40 (piccolo)', camp_small), ('n=500 (grande)', camp_large)]:
    sw_stat, sw_p  = shapiro(samp)
    ks_stat, ks_p  = kstest(samp, 'norm', args=(samp.mean(), samp.std(ddof=1)))
    jb_stat, jb_p  = jarque_bera(samp)
    print(f'\n{label}:')
    print(f'  Shapiro-Wilk:        W={sw_stat:.4f},  p={sw_p:.5f}  -> {"Normale" if sw_p>0.05 else "NON normale"}')
    print(f'  Kolmogorov-Smirnov:  D={ks_stat:.4f},  p={ks_p:.5f}  -> {"Normale" if ks_p>0.05 else "NON normale"}')
    print(f'  Jarque-Bera:         JB={jb_stat:.3f}, p={jb_p:.5f}  -> {"Normale" if jb_p>0.05 else "NON normale"}')
    print(f'  Skewness={samp.skew():.3f}, Kurtosi={samp.kurt():.3f}')

# Q-Q Plot e istogramma
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for i, (label, samp) in enumerate([('n=40', camp_small), ('n=500', camp_large)]):
    ax_hist = axes[i, 0]; ax_qq = axes[i, 1]
    ax_hist.hist(samp, bins=25, density=True, color='#3498db',
                 edgecolor='white', alpha=0.85)
    x_plot = np.linspace(samp.min(), samp.max(), 300)
    ax_hist.plot(x_plot, norm.pdf(x_plot, samp.mean(), samp.std(ddof=1)),
                 'r-', linewidth=2.5, label='N(mu,s^2) teorica')
    ax_hist.set_title(f'Istogramma SHOT_DISTANCE ({label})')
    ax_hist.set_xlabel('Distanza (piedi)'); ax_hist.set_ylabel('Densita')
    ax_hist.legend()
    (osserv, teorici), (slope, intercept, r) = stats.probplot(samp, dist='norm')
    ax_qq.scatter(teorici, osserv, s=20, alpha=0.6, color='#2ecc71')
    x_line = np.array([teorici.min(), teorici.max()])
    ax_qq.plot(x_line, slope*x_line + intercept, 'r-', linewidth=2)
    ax_qq.set_xlabel('Quantili teorici N(0,1)')
    ax_qq.set_ylabel('Quantili osservati')
    ax_qq.set_title(f'Q-Q Plot ({label}, r={r:.4f})')
plt.suptitle('Test di Normalita — SHOT_DISTANCE NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Il Q-Q plot mostra come i quantili reali si discostino da quelli normali. La deviazione nelle code rivela la distribuzione bimodale di `SHOT_DISTANCE` (picco sui tiri vicini + picco sui tiri da 3). Per $n=500$, anche Shapiro la rileva come non normale.

---

## 8.2 — Test Chi-Quadrato

### 8.2a — Bontà di adattamento (goodness-of-fit)

Verifica se una distribuzione osservata coincide con una teorica:
$$\chi^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i} \sim \chi^2_{k-1-p}$$
dove $O_i$ = frequenze osservate, $E_i$ = frequenze attese, $p$ = parametri stimati.

### 8.2b — Test di indipendenza

Verifica se due variabili categoriali sono indipendenti:
$$H_0: \text{le variabili sono indipendenti} \qquad \chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$
$$E_{ij} = \frac{R_i \cdot C_j}{N}, \quad df = (r-1)(c-1)$$

In [ ]:
# ── 8.2 Test Chi-Quadrato ──────────────────────────────────────────────────
from scipy.stats import chi2_contingency, chisquare

# --- 8.2a Bontà di adattamento: QUARTER distribuito uniformemente? ---
df24 = df[df['SEASON_1']==2024]
obs_quarter = df24['QUARTER'].value_counts().sort_index()
# Consideriamo solo i 4 quarti (escludiamo overtime)
obs_q = obs_quarter[obs_quarter.index.isin([1,2,3,4])]
n_total = obs_q.sum()
exp_q = np.array([n_total/4]*4)  # distribuzione uniforme attesa

chi2_gof, p_gof = chisquare(obs_q.values, f_exp=exp_q)
df_gof = len(obs_q) - 1

print('=== Chi-Quadrato: Bontà di adattamento ===')
print('H0: i tiri sono distribuiti uniformemente tra i 4 quarti')
print(f'\nFrequenze osservate e attese:')
for q, o, e in zip(obs_q.index, obs_q.values, exp_q):
    print(f'  Q{q}: O={o:,}, E={e:.0f}, (O-E)^2/E = {(o-e)**2/e:.2f}')
print(f'\nchi^2_obs = {chi2_gof:.4f},  df = {df_gof}')
print(f'chi^2_crit (0.05) = {chi2.ppf(0.95, df_gof):.4f}')
print(f'p-value = {p_gof:.5f}')
print(f'Decisione: {"RIFIUTO H0" if p_gof<0.05 else "non rifiuto H0"}')

# --- 8.2b Indipendenza: SHOT_TYPE e POSITION_GROUP sono indipendenti? ---
print('\n=== Chi-Quadrato: Test di indipendenza ===')
print('H0: SHOT_TYPE e POSITION_GROUP sono indipendenti')
ct = pd.crosstab(df24['POSITION_GROUP'], df24['SHOT_TYPE'])
ct = ct.loc[ct.index.isin(['G','F','C'])]
print('\nTabella di contingenza (osservata):')
print(ct)
chi2_ind, p_ind, df_ind, exp_ind = chi2_contingency(ct)
print(f'\nchi^2_obs = {chi2_ind:.4f},  df = {df_ind}')
print(f'p-value = {p_ind:.2e}')
print(f'Decisione: {"RIFIUTO H0" if p_ind<0.05 else "non rifiuto H0"}')

# Residui standardizzati
residui = (ct.values - exp_ind) / np.sqrt(exp_ind)
res_df = pd.DataFrame(residui, index=ct.index, columns=ct.columns)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Grafico bontà di adattamento
bars = ax1.bar(['Q1','Q2','Q3','Q4'], obs_q.values, color='#3498db',
               edgecolor='black', alpha=0.85, label='Osservato')
ax1.axhline(exp_q[0], color='red', linestyle='--', linewidth=2, label=f'Atteso (uniforme)={exp_q[0]:.0f}')
ax1.set_title(f'Distribuzione tiri per quarto\nchi^2={chi2_gof:.2f}, p={p_gof:.4f}')
ax1.set_ylabel('Numero di tiri'); ax1.legend()
# Heatmap residui standardizzati
sns.heatmap(res_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax2, vmin=-10, vmax=10)
ax2.set_title(f'Residui standardizzati — Indipendenza\n(POSITION vs SHOT_TYPE, p={p_ind:.2e})')
plt.suptitle('Test Chi-Quadrato — NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

print('\nResidui > |2| suggeriscono celle che contribuiscono maggiormente al chi^2:')
print(res_df.round(2))

**Interpretazione:** I residui standardizzati mostrano quali celle della tabella di contingenza si discostano di più dall'ipotesi di indipendenza. Valori $|r| > 2$ indicano celle 'anomale'. I Guards sparano molti più tiri da 3 del previsto, i Centers molto meno — non sorprendente!

---

## 8.3 — Test di Omoschedasticità

Molti test (t-test pooled, ANOVA) assumono **varianze uguali** tra i gruppi. I test principali per verificarlo:

- **Levene**: robusto, non richiede normalità (raccomandato)
- **Bartlett**: più potente ma sensibile alla non-normalità
- **F-test**: per due soli gruppi

$$H_0: \sigma_1^2 = \sigma_2^2 = \ldots = \sigma_k^2$$

In [ ]:
# ── 8.3 Test di Omoschedasticità ──────────────────────────────────────────
# Varianza di SHOT_DISTANCE uguale tra Guards, Forwards, Centers?
df24 = df[df['SEASON_1']==2024]
g_dist = df24[df24['POSITION_GROUP']=='G']['SHOT_DISTANCE'].dropna().astype(float)
f_dist = df24[df24['POSITION_GROUP']=='F']['SHOT_DISTANCE'].dropna().astype(float)
c_dist = df24[df24['POSITION_GROUP']=='C']['SHOT_DISTANCE'].dropna().astype(float)

print('=== Test di Omoschedasticità — SHOT_DISTANCE per posizione ===')
print(f'H0: Var_G = Var_F = Var_C')
print(f'\nVarianze campionarie:')
for label, d in [('Guards',g_dist),('Forwards',f_dist),('Centers',c_dist)]:
    print(f'  {label}: s^2={d.var(ddof=1):.3f},  s={d.std(ddof=1):.3f},  n={len(d):,}')

lev_stat, lev_p  = levene(g_dist, f_dist, c_dist)
bart_stat, bart_p = bartlett(g_dist, f_dist, c_dist)

print(f'\nLevene:   W={lev_stat:.4f},  p={lev_p:.5f}  -> {"RIFIUTO H0" if lev_p<0.05 else "non rifiuto H0"}')
print(f'Bartlett: B={bart_stat:.4f},  p={bart_p:.5f}  -> {"RIFIUTO H0" if bart_p<0.05 else "non rifiuto H0"}')

# F-test per due soli gruppi (Guards vs Centers)
s1 = g_dist.var(ddof=1); s2 = c_dist.var(ddof=1)
n1 = len(g_dist); n2 = len(c_dist)
F_obs = s1/s2 if s1 >= s2 else s2/s1
df1 = (n1-1) if s1>=s2 else (n2-1)
df2 = (n2-1) if s1>=s2 else (n1-1)
p_ftest = 2 * (1 - f_dist.cdf(F_obs, df1, df2))
print(f'\nF-test Guards vs Centers: F={F_obs:.4f},  p={p_ftest:.5f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Boxplot varianze
data_dict = {'Guards': g_dist.sample(1000, random_state=0).values,
             'Forwards': f_dist.sample(1000, random_state=0).values,
             'Centers': c_dist.sample(1000, random_state=0).values}
ax1.boxplot(data_dict.values(), labels=data_dict.keys(),
            notch=True, patch_artist=True,
            boxprops=dict(facecolor='#aed6f1'),
            medianprops=dict(color='red', linewidth=2))
ax1.set_ylabel('Distanza (piedi)')
ax1.set_title('Boxplot SHOT_DISTANCE per posizione')
# Visualizzazione Levene
levs = [g_dist.std(ddof=1), f_dist.std(ddof=1), c_dist.std(ddof=1)]
pos = ['Guards', 'Forwards', 'Centers']
ax2.bar(pos, levs, color=['#3498db','#e74c3c','#2ecc71'], edgecolor='black', alpha=0.85)
ax2.set_ylabel('Deviazione standard (piedi)')
ax2.set_title(f'Deviazione std per posizione\nLevene p={lev_p:.4f}')
for i,(p,v) in enumerate(zip(pos,levs)):
    ax2.text(i, v+0.1, f's={v:.2f}', ha='center', fontweight='bold')
plt.suptitle('Omoschedasticita — SHOT_DISTANCE NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Se Levene rifiuta $H_0$, le varianze differiscono significativamente tra i gruppi e l'ANOVA classica è meno affidabile (si può usare la versione di Welch). I Centers tendono ad avere distanze di tiro meno variabili (tiri vicini).

---

## 8.4 — ANOVA a Una Via (One-Way ANOVA)

L'**ANOVA** (ANalysis Of VAriance) testa se le medie di $k \geq 2$ gruppi sono uguali:
$$H_0: \mu_1 = \mu_2 = \ldots = \mu_k \qquad H_1: \text{almeno una } \mu_i \text{ diversa}$$

**Scomposizione della varianza totale:**
$$SS_{\text{tot}} = SS_{\text{tra}} + SS_{\text{entro}}, \quad F = \frac{SS_{\text{tra}}/(k-1)}{SS_{\text{entro}}/(N-k)} \sim F_{k-1,\, N-k}$$

**Post-hoc test** (es. Tukey HSD): dopo aver rifiutato $H_0$, identifica quali coppie di gruppi differiscono.

**Contesto NBA:** La distanza media dei tiri da 2 punti differisce tra Guards, Forwards e Centers?

In [ ]:
# ── 8.4 ANOVA One-Way e post-hoc Tukey ────────────────────────────────────
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df24 = df[df['SEASON_1']==2024]
# Usiamo solo tiri da 2 punti e le 3 posizioni principali
sub = df24[(df24['SHOT_TYPE']=='2PT Field Goal') &
           (df24['POSITION_GROUP'].isin(['G','F','C']))].copy()
g = sub[sub['POSITION_GROUP']=='G']['SHOT_DISTANCE'].dropna().astype(float)
f = sub[sub['POSITION_GROUP']=='F']['SHOT_DISTANCE'].dropna().astype(float)
c = sub[sub['POSITION_GROUP']=='C']['SHOT_DISTANCE'].dropna().astype(float)

print('=== ANOVA One-Way — SHOT_DISTANCE 2PT per posizione ===')
print(f'H0: mu_G = mu_F = mu_C')
for label, d in [('Guards',g),('Forwards',f),('Centers',c)]:
    print(f'  {label}: media={d.mean():.4f}, s={d.std(ddof=1):.4f}, n={len(d):,}')

F_obs, p_anova = f_oneway(g, f, c)
k = 3; N = len(g)+len(f)+len(c)
df_tra = k-1; df_ent = N-k
F_crit = f_dist.ppf(0.95, df_tra, df_ent)

print(f'\nF_obs   = {F_obs:.4f}')
print(f'F_crit  = F_(0.05, {df_tra}, {df_ent}) = {F_crit:.4f}')
print(f'p-value = {p_anova:.2e}')
print(f'Decisione: {"RIFIUTO H0" if p_anova<0.05 else "non rifiuto H0"}')

# Tabella ANOVA con statsmodels
sub_sm = sub[['POSITION_GROUP','SHOT_DISTANCE']].dropna()
model = ols('SHOT_DISTANCE ~ C(POSITION_GROUP)', data=sub_sm).fit()
anova_table = anova_lm(model)
print('\nTabella ANOVA:')
print(anova_table.round(4))

# Post-hoc Tukey HSD
print('\n=== Post-hoc Tukey HSD ===')
sub_sample = sub_sm.sample(min(3000, len(sub_sm)), random_state=42)
tukey = pairwise_tukeyhsd(sub_sample['SHOT_DISTANCE'], sub_sample['POSITION_GROUP'], alpha=0.05)
print(tukey)

# Grafico ANOVA
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
s_g = g.sample(1000, random_state=0); s_f = f.sample(1000, random_state=0); s_c = c.sample(1000, random_state=0)
data_plot = pd.DataFrame({'Distanza': pd.concat([s_g,s_f,s_c]),
                          'Posizione': ['G']*len(s_g)+['F']*len(s_f)+['C']*len(s_c)})
sns.violinplot(data=data_plot, x='Posizione', y='Distanza', ax=ax1, palette='Set2',
               order=['G','F','C'])
for pos, d, color in [('G',s_g,'#2ecc71'),('F',s_f,'#e74c3c'),('C',s_c,'#3498db')]:
    ax1.axhline(d.mean(), color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_title(f'Distribuzione 2PT per posizione\nF={F_obs:.2f}, p={p_anova:.2e}')
ax1.set_ylabel('Distanza (piedi)')
# IC per le medie
means  = [g.mean(), f.mean(), c.mean()]
errors = [1.96*d.std(ddof=1)/np.sqrt(len(d)) for d in [g,f,c]]
ax2.bar(['Guards','Forwards','Centers'], means, yerr=errors, capsize=8,
        color=['#2ecc71','#e74c3c','#3498db'], alpha=0.85, edgecolor='black')
ax2.set_ylabel('Distanza media (piedi)')
ax2.set_title('Medie con IC 95% — 2PT per posizione')
plt.suptitle('ANOVA One-Way — SHOT_DISTANCE 2PT NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** L'ANOVA rifiuta $H_0$ se le medie tra i gruppi variano più di quanto ci aspettiamo dalla variabilità casuale intra-gruppo. Il post-hoc Tukey identifica quali coppie differiscono significativamente dopo aver corretto per il problema dei test multipli.

---

## 8.5 — Correlazione: Pearson e Spearman

**Correlazione di Pearson** (lineare, per variabili continue):
$$r = \frac{\sum(X_i-\bar{X})(Y_i-\bar{Y})}{\sqrt{\sum(X_i-\bar{X})^2\sum(Y_i-\bar{Y})^2}} \in [-1, 1]$$

**Correlazione di Spearman** (ranghi, robusta a outlier e non-linearità):
$$r_s = 1 - \frac{6\sum d_i^2}{n(n^2-1)}$$

**Test di significatività:** $H_0: \rho = 0$, statistica $T = r\sqrt{(n-2)/(1-r^2)} \sim t_{n-2}$

**Attenzione:** Correlazione $\neq$ causalità.

In [ ]:
# ── 8.5 Correlazione Pearson e Spearman ───────────────────────────────────
# Correlazione tra LOC_X, LOC_Y, SHOT_DISTANCE e SHOT_MADE
df24 = df[df['SEASON_1']==2024].copy()
vars_cont = ['LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'SHOT_MADE_INT', 'QUARTER', 'MINS_LEFT']
sub_corr = df24[vars_cont].dropna().sample(5000, random_state=42)

print('=== Matrice di correlazione di Pearson ===')
corr_pearson = sub_corr.corr(method='pearson')
print(corr_pearson.round(4))

# Correlazione SHOT_DISTANCE vs SHOT_MADE
r_p, p_r = pearsonr(sub_corr['SHOT_DISTANCE'], sub_corr['SHOT_MADE_INT'])
r_s, p_s = spearmanr(sub_corr['SHOT_DISTANCE'], sub_corr['SHOT_MADE_INT'])
print(f'\nCorrelazione SHOT_DISTANCE vs SHOT_MADE:')
print(f'  Pearson:  r={r_p:.4f},  p={p_r:.2e}')
print(f'  Spearman: r={r_s:.4f},  p={p_s:.2e}')
print(f'  Interpretazione: tiri piu lontani -> FG% minore (r negativo)')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Heatmap correlazione
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
sns.heatmap(corr_pearson, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Matrice di correlazione\n(Pearson)')
# Scatter SHOT_DISTANCE vs SHOT_MADE
# FG% media per fascia di distanza
sub_bin = sub_corr.copy()
sub_bin['dist_bin'] = pd.cut(sub_bin['SHOT_DISTANCE'], bins=range(0,36,2))
fg_by_dist = sub_bin.groupby('dist_bin')['SHOT_MADE_INT'].agg(['mean','count']).reset_index()
fg_by_dist = fg_by_dist[fg_by_dist['count'] >= 20]
mid = fg_by_dist['dist_bin'].apply(lambda x: x.mid)
axes[1].scatter(mid, fg_by_dist['mean'], s=fg_by_dist['count']/20,
                alpha=0.7, color='#3498db', edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Distanza (piedi)'); axes[1].set_ylabel('FG% media')
axes[1].set_title(f'FG% vs Distanza (Pearson r={r_p:.3f})')
# LOC_X vs LOC_Y colorato per SHOT_MADE
made = sub_corr[sub_corr['SHOT_MADE_INT']==1].sample(400, random_state=0)
miss = sub_corr[sub_corr['SHOT_MADE_INT']==0].sample(400, random_state=0)
axes[2].scatter(miss['LOC_X'], miss['LOC_Y'], s=5, alpha=0.4, color='#e74c3c', label='Errore')
axes[2].scatter(made['LOC_X'], made['LOC_Y'], s=5, alpha=0.4, color='#2ecc71', label='Canestro')
axes[2].set_xlabel('LOC_X'); axes[2].set_ylabel('LOC_Y')
axes[2].set_title('Posizione sul campo: canestro vs errore')
axes[2].legend(markerscale=3)
plt.suptitle('Analisi di Correlazione — NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** La FG% decresce chiaramente con la distanza (correlazione di Pearson negativa). La mappa dei tiri mostra pattern spaziali netti: alta FG% vicino al canestro (area ristretta), bassa ai bordi della lunetta e da 3 punti.

---

## 8.6 — Regressione Lineare Semplice e Multipla

**Modello di regressione lineare:**
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \ldots + \beta_p X_p + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

**Stima OLS** (Ordinary Least Squares):
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Misure di bontà:**
$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}, \qquad R^2_{\text{adj}} = 1 - \frac{SS_{\text{res}}/(n-p-1)}{SS_{\text{tot}}/(n-1)}$$

**Contesto NBA:** Prediciamo la `SHOT_DISTANCE` in funzione di `LOC_X`, `LOC_Y` e del tipo di tiro.

In [ ]:
# ── 8.6 Regressione Lineare ────────────────────────────────────────────────
df24 = df[df['SEASON_1']==2024].copy()
sub_reg = df24[['SHOT_DISTANCE','LOC_X','LOC_Y','SHOT_TYPE']].dropna()
sub_reg['is_3pt'] = (sub_reg['SHOT_TYPE']=='3PT Field Goal').astype(int)
sub_reg = sub_reg.sample(5000, random_state=42).reset_index(drop=True)

# Modello 1: semplice (LOC_Y solo)
X1 = sm.add_constant(sub_reg['LOC_Y'])
model1 = sm.OLS(sub_reg['SHOT_DISTANCE'], X1).fit()

# Modello 2: multiplo (LOC_X, LOC_Y, is_3pt)
X2 = sm.add_constant(sub_reg[['LOC_X','LOC_Y','is_3pt']])
model2 = sm.OLS(sub_reg['SHOT_DISTANCE'], X2).fit()

print('=== Regressione Lineare Semplice: SHOT_DISTANCE ~ LOC_Y ===')
print(f'  beta_0 (intercetta) = {model1.params[0]:.4f}')
print(f'  beta_1 (LOC_Y)      = {model1.params[1]:.4f}')
print(f'  R^2                 = {model1.rsquared:.4f}')
print(f'  R^2_adj             = {model1.rsquared_adj:.4f}')
print(f'  F-stat              = {model1.fvalue:.2f}, p={model1.f_pvalue:.2e}')

print('\n=== Regressione Lineare Multipla: SHOT_DISTANCE ~ LOC_X + LOC_Y + is_3pt ===')
print(model2.summary().tables[1])
print(f'  R^2     = {model2.rsquared:.4f}')
print(f'  R^2_adj = {model2.rsquared_adj:.4f}')

# Grafici diagnostici
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
# 1. Scatter LOC_Y vs SHOT_DISTANCE con retta OLS
axes[0,0].scatter(sub_reg['LOC_Y'], sub_reg['SHOT_DISTANCE'],
                  s=5, alpha=0.3, color='#3498db')
x_plot = np.linspace(sub_reg['LOC_Y'].min(), sub_reg['LOC_Y'].max(), 200)
axes[0,0].plot(x_plot, model1.params[0]+model1.params[1]*x_plot,
               'r-', linewidth=2.5, label=f'OLS (R^2={model1.rsquared:.3f})')
axes[0,0].set_xlabel('LOC_Y'); axes[0,0].set_ylabel('SHOT_DISTANCE')
axes[0,0].set_title('Regressione semplice: LOC_Y -> SHOT_DISTANCE')
axes[0,0].legend()
# 2. Valori fitted vs residui
residui2 = model2.resid; fitted2 = model2.fittedvalues
axes[0,1].scatter(fitted2, residui2, s=5, alpha=0.3, color='#9b59b6')
axes[0,1].axhline(0, color='red', linewidth=2)
axes[0,1].set_xlabel('Valori fitted'); axes[0,1].set_ylabel('Residui')
axes[0,1].set_title('Residui vs Fitted (Mod. multiplo)')
# 3. Q-Q plot residui
(osserv, teorici), (slope, intercept, r_qq) = stats.probplot(residui2, dist='norm')
axes[1,0].scatter(teorici, osserv, s=5, alpha=0.4, color='#2ecc71')
x_line = np.array([teorici.min(), teorici.max()])
axes[1,0].plot(x_line, slope*x_line+intercept, 'r-', linewidth=2)
axes[1,0].set_xlabel('Quantili teorici'); axes[1,0].set_ylabel('Quantili residui')
axes[1,0].set_title(f'Q-Q Plot residui (r={r_qq:.4f})')
# 4. Confronto R^2
models_lab = ['Semplice\n(LOC_Y)', 'Multiplo\n(LOC_X+Y+3pt)']
r2_vals = [model1.rsquared, model2.rsquared]
axes[1,1].bar(models_lab, r2_vals, color=['#3498db','#e74c3c'], alpha=0.85, edgecolor='black')
for i,v in enumerate(r2_vals):
    axes[1,1].text(i, v+0.005, f'R^2={v:.4f}', ha='center', fontweight='bold')
axes[1,1].set_ylabel('R^2'); axes[1,1].set_title('Confronto R^2')
axes[1,1].set_ylim(0, 1)
plt.suptitle('Regressione Lineare — SHOT_DISTANCE NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:** Il modello multiplo ha un $R^2$ più alto grazie alle variabili aggiuntive. Il Q-Q plot dei residui verifica la normalità degli errori (assunzione del modello). Il grafico Residui vs Fitted non deve mostrare pattern sistematici (se ci sono, indica eteroschedasticità o non-linearità).

---

## 8.7 — Regressione Logistica

Quando la variabile dipendente è **binaria** ($Y \in \{0,1\}$), si usa la regressione logistica:
$$P(Y=1 \mid \mathbf{X}) = \frac{1}{1+e^{-\mathbf{X}\boldsymbol{\beta}}} = \sigma(\mathbf{X}\boldsymbol{\beta})$$

**Log-odds** (logit):
$$\log\frac{P(Y=1)}{P(Y=0)} = \beta_0 + \beta_1 X_1 + \ldots + \beta_p X_p$$

**Interpretazione dei coefficienti:** $e^{\beta_j}$ = **Odds Ratio** — quanto cambia il rapporto tra probabilità di successo e insuccesso per un aumento unitario di $X_j$.

**Contesto NBA:** Prediciamo la probabilità di realizzare un tiro in funzione della distanza, del tipo di tiro e del quarto.

In [ ]:
# ── 8.7 Regressione Logistica ─────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df24 = df[df['SEASON_1']==2024].copy()
features = ['SHOT_DISTANCE','LOC_X','LOC_Y','QUARTER','MINS_LEFT']
df_lr = df24[features + ['SHOT_MADE_INT']].dropna()
df_lr['is_3pt'] = (df24.loc[df_lr.index,'SHOT_TYPE']=='3PT Field Goal').astype(int)
df_lr = df_lr.sample(10000, random_state=42)

X = df_lr[features + ['is_3pt']]
y = df_lr['SHOT_MADE_INT']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Modello logistico con statsmodels (per inferenza coefficienti)
X_sm = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_sm).fit(disp=False)
print(logit_model.summary2().tables[1].round(4))

# Odds Ratio
print('\n=== Odds Ratio ===')
or_df = pd.DataFrame({
    'Coeff': logit_model.params,
    'Odds Ratio': np.exp(logit_model.params),
    'OR CI low': np.exp(logit_model.conf_int()[0]),
    'OR CI high': np.exp(logit_model.conf_int()[1]),
    'p-value': logit_model.pvalues
}).round(4)
print(or_df)

# Predizioni con sklearn per ROC
lr = LogisticRegression(random_state=42, max_iter=300)
lr.fit(X_train_sc, y_train)
y_proba = lr.predict_proba(X_test_sc)[:,1]
y_pred  = lr.predict(X_test_sc)

auc = roc_auc_score(y_test, y_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

print(f'\nAUC-ROC = {auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Errore','Canestro']))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# ROC curve
axes[0].plot(fpr, tpr, '#3498db', linewidth=2.5, label=f'AUC={auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', linewidth=1.5, label='Random')
axes[0].set_xlabel('FPR (1-Specificity)'); axes[0].set_ylabel('TPR (Sensitivity)')
axes[0].set_title('Curva ROC — Regressione Logistica')
axes[0].legend(); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)
# Probabilità predette per distanza
dist_range = np.linspace(0, 35, 200).reshape(-1,1)
X_plot_2pt = np.column_stack([dist_range, np.zeros((200,5))])  # all zeros for other features
X_plot_2pt_sc = scaler.transform(X_plot_2pt)
prob_2pt = lr.predict_proba(X_plot_2pt_sc)[:,1]
axes[1].plot(dist_range, prob_2pt, 'b-', linewidth=2.5, label='P(canestro)')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='0.5')
axes[1].set_xlabel('Distanza (piedi)'); axes[1].set_ylabel('P(SHOT_MADE=1)')
axes[1].set_title('Curva logistica: P(canestro) vs distanza')
axes[1].legend()
# Odds Ratio chart
or_plot = or_df.drop('const').copy()
colors_or = ['#e74c3c' if or_val < 1 else '#2ecc71' for or_val in or_plot['Odds Ratio']]
axes[2].barh(or_plot.index, or_plot['Odds Ratio'],
             xerr=[or_plot['Odds Ratio']-or_plot['OR CI low'],
                   or_plot['OR CI high']-or_plot['Odds Ratio']],
             color=colors_or, alpha=0.85, edgecolor='black', capsize=5)
axes[2].axvline(1, color='black', linestyle='--', linewidth=2, label='OR=1 (nessun effetto)')
axes[2].set_xlabel('Odds Ratio'); axes[2].set_title('Odds Ratio con IC 95%')
axes[2].legend(fontsize=9)
plt.suptitle('Regressione Logistica — P(SHOT_MADE) NBA 2024', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretazione:**
- La curva logistica mostra come $P(\text{canestro})$ diminuisca con la distanza
- Un **Odds Ratio > 1** per un predittore aumenta la probabilità di canestro
- Un **Odds Ratio < 1** la diminuisce
- **AUC-ROC**: 0.5 = modello casuale, 1.0 = modello perfetto. Il nostro modello è modesto (la distanza da sola non cattura tutto — serve più feature engineering)

---

## Riepilogo Dispensa 8

| Argomento | Metodo | Applicazione NBA |
|---|---|---|
| **Normalità** | Shapiro-Wilk, KS, Jarque-Bera, Q-Q | SHOT_DISTANCE bimodale |
| **Chi-Quadrato GOF** | $\chi^2 = \sum(O-E)^2/E$ | Tiri uniformi per quarto? |
| **Chi-Quadrato Indip.** | Tabella contingenza | POSITION vs SHOT_TYPE |
| **Omoschedasticità** | Levene, Bartlett | Varianza distanza per posizione |
| **ANOVA** | $F = MS_{tra}/MS_{ent}$ | Distanza 2PT: G vs F vs C |
| **Post-hoc Tukey** | HSD corretto per multipli | Coppie di posizioni diverse |
| **Correlazione Pearson** | $r \in [-1,1]$ | Distanza vs FG% |
| **Correlazione Spearman** | Ranghi, robusta | Variabili ordinali/outlier |
| **Regressione Lineare** | OLS, $R^2$, diagnostiche | Distanza ~ LOC_X + LOC_Y |
| **Regressione Logistica** | Logit, Odds Ratio, AUC | P(canestro) ~ features |

---
*Fine Dispensa 8 — Fine del Corso di Modelli Statistici*

---

## Appendice: Mappa del Corso

| Dispensa | Argomenti | Dati NBA usati |
|---|---|---|
| **D1** | Bernoulli, Binomiale, CDF, Normale/Poisson, Z-score | SHOT_MADE, SHOT_DISTANCE |
| **D2** | LGN, Distrib. campionaria, TLC, Stime, Campionamento | SHOT_MADE, SHOT_DISTANCE |
| **D3** | t-Student, Chi-Quadrato, IC Z e t, Ampiezza n | SHOT_DISTANCE, 3PT distance |
| **D4** | Proporzione p_hat, IC per p, H0/H1, Z-test | FG% per posizione |
| **D5** | Errori I/II, Potenza, Z/T stat, Uni/bilaterale | FG%, distanza 2025 |
| **D6** | Esercizi IC, Z-test, T-test, calcolo beta | Guards/Centers/stagioni |
| **D7** | P-value, T-test 1 campione, 2 campioni, appaiato | FG% stagioni, 2024 vs 2022 |
| **D8** | Normalità, Chi-Sq, Levene, ANOVA, Corr, Regressione | Tutte le variabili |
